# Notebook 04 Controlled Follow-Up - Validation Only

Protocol: `docs/CONTROLLED_FOLLOWUP_PROTOCOL_2026-06-04.md`

Scope: `validation_only`

Research question:

```text
Given the official Stage 0 candidate and Notebook 03 selected-branch context,
does a fixed small follow-up panel still show validation-only signal under
fresh seeds, and does any model show useful within-model selective confidence
structure?
```

Official candidate:

```text
candidate_id  = stage0_official
label_config  = h03_bps1p5
feature_set   = price_volume_time
window_size   = 20
```

Fixed model panel:

```text
stratified_dummy
always_up_dummy
logreg
lightgbm
standalone_tcn
ms_dlinear_tcn
```

Fresh seeds:

```text
606, 707, 808, 909, 1010
```

Hard boundaries:

- no project helper package, prior notebook, or archived helper is imported as
  active logic;
- no holdout/test rows are loaded, transformed, windowed, scored, summarized,
  displayed, or used for wording decisions;
- 04S and 04A do not fit real models;
- 04B fits only the fixed panel on the official candidate and writes per-sample
  validation prediction artifacts;
- 04C reads only saved 04B prediction artifacts and evaluates within-model
  selective coverage;
- 04D writes a manual decision matrix and does not authorize any next notebook;
- 04E is optional and reads only saved prediction artifacts.

Run profile:

```text
04S schema smoke: CPU
04A read-only context check: CPU
04B fresh-seed panel: T4 recommended
04C selective coverage: CPU after artifacts exist
04D manual gate decision: CPU after artifacts exist
04E bootstrap CI: optional, CPU after artifacts exist
```

Tabular-only completion is a partial diagnostic, not a completed Notebook 04
gate.


In [ ]:
from pathlib import Path
from collections import Counter
import importlib
import json
import math
import random
import shutil
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 160)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

INSTALL_LIGHTGBM_IF_MISSING = False
INSTALL_TORCH_IF_MISSING = False
PYTHON_DEPS_DIR = Path("/content/stage0_python_deps")


def install_package_to_local_target(package_name):
    target_dir = PYTHON_DEPS_DIR
    target_dir.mkdir(parents=True, exist_ok=True)
    base_name = package_name.split("[", 1)[0].replace("-", "_")
    for pattern in (base_name, f"{base_name}-*.dist-info"):
        for path in target_dir.glob(pattern):
            if path.is_dir():
                shutil.rmtree(path)
            elif path.exists():
                path.unlink()
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--upgrade",
        "--target",
        str(target_dir),
        package_name,
    ]
    print("Running dependency install:", " ".join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr, file=sys.stderr)
    if proc.returncode != 0:
        raise RuntimeError(
            f"pip install failed for {package_name} with exit code {proc.returncode}. "
            "Read the pip output above. If Colab reports a package or filesystem "
            "error, restart the runtime and rerun setup cells."
        )
    target_text = str(target_dir)
    if target_text not in sys.path:
        sys.path.insert(0, target_text)
    importlib.invalidate_caches()


def ensure_lightgbm():
    try:
        import lightgbm as lgb
        return lgb
    except (ImportError, OSError) as exc:
        if INSTALL_LIGHTGBM_IF_MISSING:
            print(f"LightGBM import failed before install: {type(exc).__name__}: {exc}")
            install_package_to_local_target("lightgbm")
            sys.modules.pop("lightgbm", None)
            try:
                import lightgbm as lgb
                return lgb
            except (ImportError, OSError) as retry_exc:
                raise RuntimeError(
                    "LightGBM still cannot import after installing into "
                    f"{PYTHON_DEPS_DIR}. Restart the Colab runtime and rerun "
                    "setup cells before Stage 0S."
                ) from retry_exc
        raise RuntimeError(
            "LightGBM import failed. Set INSTALL_LIGHTGBM_IF_MISSING=True and "
            "rerun setup cells. The notebook will install LightGBM into "
            f"{PYTHON_DEPS_DIR} and place that directory first on sys.path."
        ) from exc


def ensure_torch():
    try:
        import torch
        return torch
    except (ImportError, OSError) as exc:
        if INSTALL_TORCH_IF_MISSING:
            print(f"PyTorch import failed before install: {type(exc).__name__}: {exc}")
            install_package_to_local_target("torch")
            sys.modules.pop("torch", None)
            try:
                import torch
                return torch
            except (ImportError, OSError) as retry_exc:
                raise RuntimeError(
                    "PyTorch still cannot import after installing into "
                    f"{PYTHON_DEPS_DIR}. Restart the Colab runtime and rerun "
                    "setup cells before Stage 0B."
                ) from retry_exc
        raise RuntimeError(
            "PyTorch import failed. Set INSTALL_TORCH_IF_MISSING=True and rerun "
            "setup cells before enabling Stage 0B deep models."
        ) from exc


In [ ]:
\
TICKERS = ("CSCO", "JPM", "KO", "MSFT", "WMT")
FRESH_SEEDS = (606, 707, 808, 909, 1010)
MODEL_SEEDS = FRESH_SEEDS
RESULT_SCOPE = "validation_only"

INSTALL_LIGHTGBM_IF_MISSING = False
INSTALL_TORCH_IF_MISSING = False

RUN_04S_SCHEMA_SMOKE = False
RUN_04A_READ_CONTEXT = False
RUN_04B_FRESH_SEED_PANEL = False
RUN_04C_SELECTIVE_COVERAGE = False
RUN_04D_GATE_DECISION = False
RUN_04E_BOOTSTRAP_CI = False
BACKUP_NOTEBOOK04_TO_GOOGLE_DRIVE = False

DRIVE_PROJECT_FOLDER_ID = "15IZ_sOEyyAKmGCUIOv_u17SwQmFX3nG_"
NOTEBOOK03_DRIVE_RESULTS_FOLDER_ID = "1qQbkwV07X6L_D_WtRYrHDmZ3KXjsju9r"
NOTEBOOK03_DRIVE_RESULTS_FOLDER_NAME = "notebook03_model_family_screening_results"
NOTEBOOK04_DRIVE_BACKUP_FOLDER_NAME = "notebook04_controlled_followup_results"

BOOTSTRAP_RESAMPLES = 1000
BOOTSTRAP_SEED = 260604
SELECTIVE_COVERAGE_GRID = (1.00, 0.80, 0.60, 0.40, 0.20, 0.10)
MIN_PRACTICAL_DELTA_MACRO_F1 = 0.005

NOTEBOOK04_CANDIDATE = {
    "candidate_id": "stage0_official",
    "label_config": "h03_bps1p5",
    "feature_set": "price_volume_time",
    "window_size": 20,
    "source": "official_stage0_candidate_from_notebook02",
}

BASELINE_MODELS = ("stratified_dummy", "always_up_dummy")
TABULAR_MODELS = ("logreg", "lightgbm")
SEQUENCE_MODELS = ("standalone_tcn", "ms_dlinear_tcn")
REAL_MODELS = TABULAR_MODELS + SEQUENCE_MODELS
MODEL_PANEL = BASELINE_MODELS + REAL_MODELS

ALL_FEATURES = (
    "log_return",
    "close_to_open_return",
    "high_low_range",
    "rolling_volatility_20",
    "normalized_volume_20",
    "rsi_14",
    "bollinger_pctb",
    "normalized_macd_hist",
    "time_of_day_sin",
    "time_of_day_cos",
)

FEATURE_SETS = {
    "price_action_core": (
        "log_return",
        "close_to_open_return",
        "high_low_range",
    ),
    "technical_price": (
        "log_return",
        "high_low_range",
        "rsi_14",
        "bollinger_pctb",
        "normalized_macd_hist",
    ),
    "price_volume_time": ALL_FEATURES,
}

LABEL_CONFIGS = {
    "h03_bps1p5": {"horizon_k": 3, "threshold_bps": 1.5},
    "h09_bps3p0": {"horizon_k": 9, "threshold_bps": 3.0},
    "h24_bps7p5": {"horizon_k": 24, "threshold_bps": 7.5},
}

MAX_TRAIN_ROWS = None
RANDOM_SUBSAMPLE_SEED = 42

LGBM_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.03,
    "max_depth": 6,
    "num_leaves": 31,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
}

TORCH_EPOCHS = 8
TORCH_BATCH_SIZE = 1024
TORCH_LEARNING_RATE = 1e-3
TORCH_WEIGHT_DECAY = 1e-4
TORCH_TCN_CHANNELS = (32, 32)
TORCH_TCN_KERNEL_SIZE = 3
TORCH_MOVING_AVG_KERNELS = (3, 5, 9, 15)
TORCH_DROPOUT = 0.10

TRAIN_START, TRAIN_END = "1998-01-02", "2013-09-16"
VAL_START, VAL_END = "2013-09-16", "2017-01-25"
CALENDAR_SPLITS = {
    "train": (TRAIN_START, TRAIN_END),
    "validation": (VAL_START, VAL_END),
}

BPS_TO_DECIMAL = 10000.0
BAR_INTERVAL_MINUTES = 5
MARKET_OPEN_MINUTE = 9 * 60 + 30
TRADING_DAY_MINUTES = 390
TIME_OF_DAY_ENCODING_PERIOD_MINUTES = TRADING_DAY_MINUTES + BAR_INTERVAL_MINUTES
MARKET_OPEN = pd.Timestamp("09:30").time()
MARKET_CLOSE = pd.Timestamp("16:00").time()
EXPECTED_COLUMNS = ("timestamp", "open", "high", "low", "close", "volume")
DATA_FILE_SUFFIXES = (".csv", ".txt")
RAW_TXT_COLUMNS = ("Date", "Time", "Open", "High", "Low", "Close", "Volume")

OUTPUT_DIR = Path("/content/notebook04_controlled_followup_results")
PREDICTION_DIR = OUTPUT_DIR / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILES = {
    "context": OUTPUT_DIR / "notebook04_context_checks.json",
    "pooled": OUTPUT_DIR / "notebook04_pooled.csv",
    "per_ticker": OUTPUT_DIR / "notebook04_per_ticker.csv",
    "summary": OUTPUT_DIR / "notebook04_summary.csv",
    "prediction_manifest": OUTPUT_DIR / "notebook04_prediction_manifest.csv",
    "run_manifest": OUTPUT_DIR / "notebook04_run_manifest.json",
    "selective": OUTPUT_DIR / "notebook04_selective_coverage.csv",
    "decision": OUTPUT_DIR / "notebook04_decision_matrix.csv",
    "bootstrap": OUTPUT_DIR / "notebook04_bootstrap_ci.csv",
}

NOTEBOOK03_SELECTION_CANDIDATES = (
    Path("/content/notebook03_model_family_screening_results/notebook03_validation_selection.json"),
    Path("/content/notebook03_validation_selection.json"),
)
NOTEBOOK03_SUMMARY_CANDIDATES = (
    Path("/content/notebook03_model_family_screening_results/notebook03_summary.csv"),
    Path("/content/notebook03_summary.csv"),
)
H0_SUMMARY_CANDIDATES = (
    Path("/content/diagnostic_h0_tabular_sweep/diagnostic_h0_summary.csv"),
    Path("/content/diagnostic_h0_summary.csv"),
)

NOTEBOOK04_STATE = {"prediction_manifest_rows": []}

display(pd.DataFrame([NOTEBOOK04_CANDIDATE]))
print("Notebook 04 output directory:", OUTPUT_DIR)
print("Notebook 03 Drive results folder:", NOTEBOOK03_DRIVE_RESULTS_FOLDER_NAME)
print("Notebook 04 Drive backup folder:", NOTEBOOK04_DRIVE_BACKUP_FOLDER_NAME)
print("Model panel:", MODEL_PANEL)
print("Fresh seeds:", FRESH_SEEDS)
print("Run switches:", {
    "RUN_04S_SCHEMA_SMOKE": RUN_04S_SCHEMA_SMOKE,
    "RUN_04A_READ_CONTEXT": RUN_04A_READ_CONTEXT,
    "RUN_04B_FRESH_SEED_PANEL": RUN_04B_FRESH_SEED_PANEL,
    "RUN_04C_SELECTIVE_COVERAGE": RUN_04C_SELECTIVE_COVERAGE,
    "RUN_04D_GATE_DECISION": RUN_04D_GATE_DECISION,
    "RUN_04E_BOOTSTRAP_CI": RUN_04E_BOOTSTRAP_CI,
    "BACKUP_NOTEBOOK04_TO_GOOGLE_DRIVE": BACKUP_NOTEBOOK04_TO_GOOGLE_DRIVE,
})


## Data Loading

This cell resolves the five real ticker files and downloads them through the
Google Drive API when needed. It does not mount MyDrive. For raw `.txt` files it
first scans to the validation boundary and then reads only rows before
`VAL_END`, so closed holdout/test rows are not materialized into the notebook
dataframes. Outputs stay local under `/content/stage0_config_screening_results`.


In [ ]:
RAW_DRIVE_FOLDER_ID = "154SlcH3nViUcvPXFBM-E4NPg_ybljBTG"
RAW_DRIVE_FOLDER_NAME = "s&p 100 adjusted 1 min data"
RAW_DATA_DIR = Path("/content/stage0_raw_stock_data")
DOWNLOAD_RAW_DATA_FROM_DRIVE = True

RAW_DRIVE_FILES = {
    "CSCO": {"name": "CSCO.txt", "file_id": "17A49kUiMELuQqdkOhw1KrpudjP5i5xIN"},
    "JPM": {"name": "JPM.txt", "file_id": "11UQUJKVXTrBb8XFWY5Z8JDQ8_4i_DE-q"},
    "KO": {"name": "KO.txt", "file_id": "1XmtwuZ2dTP20NsU27w5dMyRdSvdnNTSn"},
    "MSFT": {"name": "MSFT.txt", "file_id": "1Ud1SQpQbaiRKemFf9dgu1o_raUPnFvGs"},
    "WMT": {"name": "WMT.txt", "file_id": "1NNfsoUJrrsj2ae5EnC-PTPcZs_QGR_7c"},
}


def is_real_raw_file(path: Path) -> bool:
    if not path.is_file():
        return False
    if path.name.lower().endswith(".gshortcut"):
        return False
    if path.suffix.lower() not in DATA_FILE_SUFFIXES:
        return False
    try:
        return path.stat().st_size > 50_000
    except OSError:
        return False


def build_drive_service():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError(
            "Drive API is unavailable. Open this notebook in Google Colab and "
            "authenticate when prompted; do not use drive.mount for Stage 0 data."
        ) from exc
    auth.authenticate_user()
    return build("drive", "v3")


def download_raw_drive_files():
    if not DOWNLOAD_RAW_DATA_FROM_DRIVE:
        return
    try:
        from googleapiclient.http import MediaIoBaseDownload
    except ImportError as exc:
        raise RuntimeError("googleapiclient is unavailable in this Colab runtime.") from exc

    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    service = build_drive_service()
    for ticker, item in RAW_DRIVE_FILES.items():
        target = RAW_DATA_DIR / item["name"]
        if is_real_raw_file(target):
            print(f"{ticker}: using cached raw file {target}")
            continue
        print(f"{ticker}: downloading raw Drive file {item['name']} -> {target}")
        request = service.files().get_media(fileId=item["file_id"])
        with target.open("wb") as output:
            downloader = MediaIoBaseDownload(output, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        if not is_real_raw_file(target):
            raise ValueError(f"Downloaded file is not a real raw ticker file: {target}")


def resolve_data_files():
    if DOWNLOAD_RAW_DATA_FROM_DRIVE:
        download_raw_drive_files()
    files = {}
    missing = []
    for ticker, item in RAW_DRIVE_FILES.items():
        path = RAW_DATA_DIR / item["name"]
        if is_real_raw_file(path):
            files[ticker] = path
        else:
            missing.append(f"{ticker}: {path}")
    if missing:
        raise FileNotFoundError(
            "Missing required raw ticker files after Drive API resolution: "
            + "; ".join(missing)
        )
    print("resolved raw Drive data files:")
    for ticker, path in files.items():
        print(f"  {ticker}: {path}")
    return files


def find_timestamp_column(columns):
    for candidate in ("timestamp", "datetime", "date", "time"):
        for column in columns:
            if str(column).lower() == candidate:
                return column
    raise ValueError(f"No timestamp-like column found in columns: {list(columns)}")


def normalize_ohlcv_columns(frame, source_name):
    lower_map = {str(column).lower(): column for column in frame.columns}
    rename = {}
    for required in EXPECTED_COLUMNS:
        if required == "timestamp":
            continue
        if required not in lower_map:
            raise ValueError(f"{source_name} missing required column: {required}")
        rename[lower_map[required]] = required
    return frame.rename(columns=rename)


def resample_to_five_minutes(frame):
    resampled = (
        frame.set_index("timestamp")
        .resample("5min")
        .agg({"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"})
        .dropna(subset=["open", "high", "low", "close", "volume"])
        .reset_index()
    )
    times = resampled["timestamp"].dt.time
    return resampled.loc[
        (times >= MARKET_OPEN) & (times <= MARKET_CLOSE),
        list(EXPECTED_COLUMNS),
    ].reset_index(drop=True)


def txt_date_key(date_text):
    parts = str(date_text).strip().split("/")
    if len(parts) != 3:
        raise ValueError(f"Unexpected Date field in raw txt file: {date_text!r}")
    month, day, year = parts
    return int(year), int(month), int(day)


def count_txt_rows_before_validation_end(path):
    validation_end_key = txt_date_key(pd.Timestamp(VAL_END).strftime("%m/%d/%Y"))
    safe_rows = 0
    has_header = False
    reached_boundary = False
    with Path(path).open("rt", encoding="utf-8", errors="replace", newline="") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped:
                continue
            first_field = stripped.split(",", 1)[0].strip()
            if first_field.lower() == "date":
                has_header = True
                continue
            if txt_date_key(first_field) >= validation_end_key:
                reached_boundary = True
                break
            safe_rows += 1
    if safe_rows == 0:
        raise ValueError(f"No train/validation rows found before {VAL_END} in: {path}")
    if not reached_boundary:
        print(f"{Path(path).name}: no row at or after {VAL_END} found; read capped to file end.")
    return safe_rows, has_header


def read_one_minute_txt(path):
    safe_rows, has_header = count_txt_rows_before_validation_end(path)
    print(f"{Path(path).name}: loading {safe_rows:,} raw one-minute rows before {VAL_END}.")
    frame = pd.read_csv(
        path,
        header=None,
        names=RAW_TXT_COLUMNS,
        nrows=safe_rows,
        skiprows=1 if has_header else None,
        low_memory=False,
    )
    frame = frame.loc[frame["Date"].astype(str).str.lower() != "date"].reset_index(drop=True)
    frame["timestamp"] = pd.to_datetime(
        frame["Date"].astype(str) + " " + frame["Time"].astype(str),
        format="%m/%d/%Y %H:%M",
        errors="raise",
    )
    frame = frame.drop(columns=["Date", "Time"]).rename(
        columns={"Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"}
    )
    numeric_columns = ["open", "high", "low", "close", "volume"]
    frame[numeric_columns] = frame[numeric_columns].apply(pd.to_numeric, errors="raise")
    validation_end = pd.Timestamp(VAL_END)
    times = frame["timestamp"].dt.time
    frame = frame.loc[
        (frame["timestamp"] < validation_end)
        & (times >= MARKET_OPEN)
        & (times <= MARKET_CLOSE),
        list(EXPECTED_COLUMNS),
    ]
    return resample_to_five_minutes(frame)


def read_five_minute_csv(path):
    validation_end = pd.Timestamp(VAL_END)
    chunks = []
    for chunk in pd.read_csv(path, chunksize=100_000):
        timestamp_column = find_timestamp_column(chunk.columns)
        chunk = chunk.rename(columns={timestamp_column: "timestamp"}).copy()
        chunk = normalize_ohlcv_columns(chunk, path.name)
        chunk["timestamp"] = pd.to_datetime(chunk["timestamp"], errors="raise")
        raw_chunk_max_timestamp = chunk["timestamp"].max()
        chunk = chunk.loc[chunk["timestamp"] < validation_end, list(EXPECTED_COLUMNS)]
        if not chunk.empty:
            chunks.append(chunk)
        if raw_chunk_max_timestamp >= validation_end:
            break
    if not chunks:
        raise ValueError(f"No train/validation rows found in: {path}")
    return pd.concat(chunks, ignore_index=True)


def load_ticker(ticker, path):
    path = Path(path)
    frame = read_one_minute_txt(path) if path.suffix.lower() == ".txt" else read_five_minute_csv(path)
    frame["ticker"] = ticker
    return frame.sort_values("timestamp").reset_index(drop=True)


RUN_ANY_STAGE = bool(RUN_04B_FRESH_SEED_PANEL)

if RUN_ANY_STAGE:
    DATA_FILES = resolve_data_files()
    raw_data = {ticker: load_ticker(ticker, DATA_FILES[ticker]) for ticker in TICKERS}

    display(pd.DataFrame([
        {
            "ticker": ticker,
            "rows": len(frame),
            "start": frame["timestamp"].min(),
            "end": frame["timestamp"].max(),
            "source": DATA_FILES[ticker].name,
            "path": str(DATA_FILES[ticker]),
        }
        for ticker, frame in raw_data.items()
    ]))
else:
    DATA_FILES = {}
    raw_data = {}
    print("All Notebook 04 data-loading switches are False; data loading skipped.")


## Feature, Label, Split, Scale, Window

These functions implement the active chronology-safe Stage 0 contracts inside a
standalone Colab notebook. They do not import a local helper package or a prior
route. Keep the causal post-bar-close feature rules, cumulative
horizon-return labels, split-boundary invalidation, train-only scaler fitting,
and per-ticker/per-day windows aligned with the active freeze documents before
rerunning Stage 0.

Tabular models use flattened windows: each LogReg/LightGBM sample is
`window_size * n_features`, built from the same per-ticker/per-day rows used by
the sequence models. This makes Stage 0A2 a true window-length sensitivity
check rather than a last-bar sample-count check.

Feature timing boundary: prediction is after the current five-minute bar has
completed, so `close[t]`, `high[t]`, `low[t]`, `volume[t]`, and same-row
timestamp encodings are available. Same-day trailing Bollinger windows may
include `close[t]`; RSI and MACD EWM states are causal but intentionally
continuous across trading days.


In [ ]:
def finite_mask(frame, columns):
    return np.isfinite(frame[list(columns)].to_numpy(dtype=float)).all(axis=1)


def grouped_rolling(series, group_key, window, how):
    grouped = series.groupby(group_key, group_keys=False)
    if how == "mean":
        return grouped.transform(lambda part: part.rolling(window, min_periods=window).mean())
    if how == "std":
        return grouped.transform(lambda part: part.rolling(window, min_periods=window).std())
    raise ValueError(f"Unknown rolling operation: {how}")


def continuous_ewm(series, span):
    return series.ewm(span=span, adjust=False, min_periods=span).mean()


def continuous_wilder_ewm(series, period):
    return series.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()


def validated_ohlcv(frame):
    out = frame[["open", "high", "low", "close", "volume"]].astype(float)
    bad = (
        (out[["open", "high", "low", "close"]] <= 0).any(axis=1)
        | (out["volume"] < 0)
        | (out["high"] < out["low"])
        | (out["open"] < out["low"])
        | (out["open"] > out["high"])
        | (out["close"] < out["low"])
        | (out["close"] > out["high"])
    )
    if bad.any():
        raise ValueError(f"Invalid OHLCV rows found: {int(bad.sum())}")
    return out


def add_features(frame):
    current = frame.sort_values("timestamp").copy()
    day = current["timestamp"].dt.date
    ohlcv = validated_ohlcv(current)
    close, open_, high, low, volume = (ohlcv[c] for c in ["close", "open", "high", "low", "volume"])

    log_close = np.log(close)
    current["log_return"] = log_close.groupby(day, group_keys=False).diff()
    current["close_to_open_return"] = close / open_ - 1.0
    current["high_low_range"] = np.log(high / low)

    shifted_log_return = current["log_return"].groupby(day, group_keys=False).shift(1)
    current["rolling_volatility_20"] = grouped_rolling(shifted_log_return, day, 20, "std")

    log_volume = np.log1p(volume)
    shifted_log_volume = log_volume.groupby(day, group_keys=False).shift(1)
    current["normalized_volume_20"] = log_volume - grouped_rolling(shifted_log_volume, day, 20, "mean")

    close_delta = close.groupby(day, group_keys=False).diff()
    gain = close_delta.clip(lower=0.0)
    loss = (-close_delta).clip(lower=0.0)
    avg_gain = continuous_wilder_ewm(gain, 14)
    avg_loss = continuous_wilder_ewm(loss, 14)
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    rsi = rsi.mask(avg_loss.eq(0.0) & avg_gain.gt(0.0), 100.0)
    current["rsi_14"] = rsi.mask(avg_loss.eq(0.0) & avg_gain.eq(0.0), 50.0)

    rolling_mean_20 = grouped_rolling(close, day, 20, "mean")
    rolling_std_20 = grouped_rolling(close, day, 20, "std")
    lower_band = rolling_mean_20 - 2.0 * rolling_std_20
    upper_band = rolling_mean_20 + 2.0 * rolling_std_20
    current["bollinger_pctb"] = (close - lower_band) / (upper_band - lower_band).replace(0.0, np.nan)

    ema_12 = continuous_ewm(close, 12)
    ema_26 = continuous_ewm(close, 26)
    macd = ema_12 - ema_26
    signal = continuous_ewm(macd, 9)
    current["normalized_macd_hist"] = (macd - signal) / ema_26.replace(0.0, np.nan)

    minute_of_day = current["timestamp"].dt.hour * 60 + current["timestamp"].dt.minute
    minutes_since_open = minute_of_day - MARKET_OPEN_MINUTE
    current["time_of_day_sin"] = np.sin(2.0 * np.pi * minutes_since_open / TIME_OF_DAY_ENCODING_PERIOD_MINUTES)
    current["time_of_day_cos"] = np.cos(2.0 * np.pi * minutes_since_open / TIME_OF_DAY_ENCODING_PERIOD_MINUTES)
    return current.replace([np.inf, -np.inf], np.nan)


def add_labels(frame, horizon_k, threshold_bps):
    current = frame.sort_values("timestamp").copy()
    threshold = threshold_bps / BPS_TO_DECIMAL
    close = current["close"].astype(float)
    future_timestamp = current["timestamp"].shift(-horizon_k)
    current["future_cumulative_return"] = close.shift(-horizon_k) / close - 1.0

    same_day = pd.Series(True, index=current.index)
    current_day = current["timestamp"].dt.date
    for offset in range(1, horizon_k + 1):
        same_day &= current_day.shift(-offset).eq(current_day)

    actual_horizon = future_timestamp - current["timestamp"]
    expected_horizon = pd.Timedelta(minutes=BAR_INTERVAL_MINUTES * horizon_k)
    current["diagnostic_irregular_horizon"] = (
        future_timestamp.notna() & same_day & actual_horizon.ne(expected_horizon)
    )
    current["invalid_irregular_horizon"] = current["diagnostic_irregular_horizon"]
    current["invalid_missing_future"] = current["future_cumulative_return"].isna()
    current["invalid_cross_day"] = ~same_day
    invalid = (
        current["invalid_missing_future"]
        | current["invalid_cross_day"]
        | current["invalid_irregular_horizon"]
    )
    current["label"] = np.nan
    valid = ~invalid
    current.loc[invalid, "future_cumulative_return"] = np.nan
    current.loc[valid & (current["future_cumulative_return"] > threshold), "label"] = 1
    current.loc[valid & (current["future_cumulative_return"] < -threshold), "label"] = 0
    return current


def assign_split(timestamp):
    ts = pd.Timestamp(timestamp)
    for split_name, (start, end) in CALENDAR_SPLITS.items():
        if pd.Timestamp(start) <= ts < pd.Timestamp(end):
            return split_name
    return "outside_train_validation"


def add_split_and_invalidate_boundaries(frame, horizon_k):
    current = frame.sort_values("timestamp").copy()
    current["split"] = current["timestamp"].map(assign_split)
    horizon_split = current["split"].shift(-horizon_k)
    cross_split = current["future_cumulative_return"].notna() & (current["split"] != horizon_split)
    current["invalid_cross_split"] = cross_split
    current.loc[cross_split, "label"] = np.nan
    current.loc[cross_split, "future_cumulative_return"] = np.nan
    return current


def prepare_split_frames(raw_frames, horizon_k, threshold_bps):
    split_frames = {}
    for ticker, frame in raw_frames.items():
        featured = add_features(frame)
        labeled = add_labels(featured, horizon_k=horizon_k, threshold_bps=threshold_bps)
        split_frames[ticker] = add_split_and_invalidate_boundaries(labeled, horizon_k=horizon_k)
    return split_frames


def fit_transform_train_validation(split_frames, feature_columns):
    train_parts = []
    for frame in split_frames.values():
        train = frame.loc[frame["split"] == "train", list(feature_columns)]
        train_parts.append(train.loc[finite_mask(train, feature_columns)])
    train_matrix = pd.concat(train_parts, axis=0)
    if train_matrix.empty:
        raise ValueError("No train rows available for scaler fit.")

    scaler = StandardScaler().fit(train_matrix)
    scaled = {}
    scaled_columns = [f"{name}_scaled" for name in feature_columns]
    for ticker, frame in split_frames.items():
        current = frame.copy()
        for col in scaled_columns:
            current[col] = np.nan
        eligible = current["split"].isin(["train", "validation"])
        complete = finite_mask(current, feature_columns)
        rows = eligible & complete
        if rows.any():
            current.loc[rows, scaled_columns] = scaler.transform(current.loc[rows, feature_columns])
        scaled[ticker] = current
    return scaled


def build_last_step_windows(frames_by_ticker, feature_columns, split_name, window_size):
    # Flatten the past `window_size` bars into a (window_size * n_features,) vector per sample.
    # Tabular models (LogReg/LightGBM) thus consume lagged history instead of only the last bar,
    # which makes window_size a meaningful search dimension for them too.
    scaled_columns = [f"{name}_scaled" for name in feature_columns]
    n_features = len(feature_columns)
    flat_dim = window_size * n_features
    x_parts, y_parts, owner_parts, timestamp_parts = [], [], [], []
    for ticker, frame in frames_by_ticker.items():
        segment = frame.loc[frame["split"] == split_name].sort_values("timestamp")
        for _, day_frame in segment.groupby(segment["timestamp"].dt.date, sort=True):
            day_frame = day_frame.sort_values("timestamp")
            values = day_frame[scaled_columns].to_numpy(dtype=float)
            labels = day_frame["label"].to_numpy()
            timestamps = day_frame["timestamp"].to_numpy()
            complete_rows = np.isfinite(values).all(axis=1)
            for end_idx in range(window_size - 1, len(day_frame)):
                start_idx = end_idx - window_size + 1
                if not complete_rows[start_idx : end_idx + 1].all():
                    continue
                if pd.isna(labels[end_idx]):
                    continue
                x_parts.append(values[start_idx : end_idx + 1].reshape(-1))
                y_parts.append(int(labels[end_idx]))
                owner_parts.append(ticker)
                timestamp_parts.append(timestamps[end_idx])

    if not x_parts:
        return (
            np.empty((0, flat_dim), dtype=float),
            np.asarray([], dtype=int),
            np.asarray([], dtype=object),
            np.asarray([], dtype="datetime64[ns]"),
        )
    return (
        np.stack(x_parts),
        np.asarray(y_parts, dtype=int),
        np.asarray(owner_parts, dtype=object),
        np.asarray(timestamp_parts, dtype="datetime64[ns]"),
    )


## Notebook 04 Base Helpers

This section copies active Stage 0 metric, dataset, tabular, and sequence helper definitions. The following cell overrides only the Notebook 04 orchestration and artifact layer.

In [ ]:
DATASET_CACHE = {}


def label_spec(label_config):
    spec = LABEL_CONFIGS[label_config]
    return int(spec["horizon_k"]), float(spec["threshold_bps"])


def subsample_rows_uniformly(x_values, y_values, max_rows, seed=RANDOM_SUBSAMPLE_SEED):
    if max_rows is None or len(y_values) <= max_rows:
        return x_values, y_values
    rng = np.random.default_rng(seed)
    selected = np.sort(rng.choice(len(y_values), size=int(max_rows), replace=False))
    return x_values[selected], y_values[selected]


def subsample_rows_with_owner(x_values, y_values, owner_values, max_rows, seed=RANDOM_SUBSAMPLE_SEED):
    if max_rows is None or len(y_values) <= max_rows:
        return x_values, y_values, owner_values
    rng = np.random.default_rng(seed)
    selected = np.sort(rng.choice(len(y_values), size=int(max_rows), replace=False))
    return x_values[selected], y_values[selected], owner_values[selected]


def evaluate_predictions(y_true, predictions):
    return {
        "macro_f1": float(f1_score(y_true, predictions, labels=[0, 1], average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "accuracy": float(accuracy_score(y_true, predictions)),
    }


def dummy_metrics(y_train, y_validation, seed):
    if len(y_train) == 0 or len(y_validation) == 0:
        return {"dummy_macro_f1": np.nan, "dummy_balanced_accuracy": np.nan}
    x_train = np.zeros((len(y_train), 1))
    x_validation = np.zeros((len(y_validation), 1))
    dummy = DummyClassifier(strategy="stratified", random_state=seed).fit(x_train, y_train)
    pred = dummy.predict(x_validation)
    return {
        "dummy_macro_f1": float(f1_score(y_validation, pred, labels=[0, 1], average="macro", zero_division=0)),
        "dummy_balanced_accuracy": float(balanced_accuracy_score(y_validation, pred)),
    }


def sample_std(values):
    values = pd.Series(values).dropna()
    return float(values.std(ddof=1)) if len(values) > 1 else 0.0


T_CRITICAL_ONE_SIDED_95 = {
    2: 6.314,
    3: 2.920,
    4: 2.353,
    5: 2.132,
    6: 2.015,
    7: 1.943,
    8: 1.895,
    9: 1.860,
    10: 1.833,
    11: 1.812,
    12: 1.796,
}


def t_critical_one_sided_95(seed_count):
    if seed_count <= 1:
        return 0.0
    return T_CRITICAL_ONE_SIDED_95.get(int(seed_count), 1.645)


def build_sequence_windows(frames_by_ticker, feature_columns, split_name, window_size):
    scaled_columns = [f"{name}_scaled" for name in feature_columns]
    x_parts, y_parts, owner_parts, timestamp_parts = [], [], [], []
    for ticker, frame in frames_by_ticker.items():
        segment = frame.loc[frame["split"] == split_name].sort_values("timestamp")
        for _, day_frame in segment.groupby(segment["timestamp"].dt.date, sort=True):
            day_frame = day_frame.sort_values("timestamp")
            values = day_frame[scaled_columns].to_numpy(dtype=float)
            labels = day_frame["label"].to_numpy()
            timestamps = day_frame["timestamp"].to_numpy()
            complete_rows = np.isfinite(values).all(axis=1)
            for end_idx in range(window_size - 1, len(day_frame)):
                start_idx = end_idx - window_size + 1
                if not complete_rows[start_idx : end_idx + 1].all():
                    continue
                if pd.isna(labels[end_idx]):
                    continue
                x_parts.append(values[start_idx : end_idx + 1])
                y_parts.append(int(labels[end_idx]))
                owner_parts.append(ticker)
                timestamp_parts.append(timestamps[end_idx])
    if not x_parts:
        return (
            np.empty((0, window_size, len(feature_columns)), dtype=float),
            np.asarray([], dtype=int),
            np.asarray([], dtype=object),
            np.asarray([], dtype="datetime64[ns]"),
        )
    return (
        np.stack(x_parts).astype(np.float32),
        np.asarray(y_parts, dtype=int),
        np.asarray(owner_parts, dtype=object),
        np.asarray(timestamp_parts, dtype="datetime64[ns]"),
    )


def get_dataset(label_config, feature_set, window_size):
    key = (label_config, feature_set, int(window_size))
    if key in DATASET_CACHE:
        dataset = DATASET_CACHE[key].copy()
        dataset["prep_seconds"] = 0.0
        return dataset
    if not raw_data:
        raise RuntimeError("raw_data is empty. Enable a RUN_STAGE0* switch and rerun data loading first.")
    horizon_k, threshold_bps = label_spec(label_config)
    feature_columns = FEATURE_SETS[feature_set]
    start = time.perf_counter()
    split_frames = prepare_split_frames(raw_data, horizon_k=horizon_k, threshold_bps=threshold_bps)
    scaled_frames = fit_transform_train_validation(split_frames, feature_columns)
    x_train, y_train, train_owner, train_timestamp = build_last_step_windows(
        scaled_frames, feature_columns, "train", window_size
    )
    x_validation, y_validation, validation_owner, validation_timestamp = build_last_step_windows(
        scaled_frames, feature_columns, "validation", window_size
    )
    x_train_seq, y_train_seq, train_owner_seq, train_timestamp_seq = build_sequence_windows(
        scaled_frames, feature_columns, "train", window_size
    )
    x_validation_seq, y_validation_seq, validation_owner_seq, validation_timestamp_seq = build_sequence_windows(
        scaled_frames, feature_columns, "validation", window_size
    )
    if len(y_train) == 0 or len(y_validation) == 0:
        raise ValueError(f"No tabular windows available for {label_config} / {feature_set} / window={window_size}")
    if len(y_train_seq) != len(y_train) or len(y_validation_seq) != len(y_validation):
        raise ValueError("Tabular and sequence window counts disagree; inspect window construction.")
    dataset = {
        "label_config": label_config,
        "horizon_k": horizon_k,
        "threshold_bps": threshold_bps,
        "feature_set": feature_set,
        "feature_columns": feature_columns,
        "window_size": int(window_size),
        "x_train": x_train,
        "y_train": y_train,
        "train_owner": train_owner,
        "x_validation": x_validation,
        "y_validation": y_validation,
        "validation_owner": validation_owner,
        "x_train_seq": x_train_seq,
        "y_train_seq": y_train_seq,
        "train_owner_seq": train_owner_seq,
        "x_validation_seq": x_validation_seq,
        "y_validation_seq": y_validation_seq,
        "validation_owner_seq": validation_owner_seq,
        "prep_seconds": time.perf_counter() - start,
    }
    DATASET_CACHE[key] = dataset.copy()
    return dataset


def fit_predict_logreg(dataset, seed):
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    # Tabular features are flattened windows, so allow more iterations for the
    # higher-dimensional design matrix.
    max_iter = 2000
    model = LogisticRegression(
        solver="liblinear",
        max_iter=max_iter,
        class_weight="balanced",
        random_state=seed,
    )
    start_fit = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    pred = model.predict(dataset["x_validation"])
    predict_seconds = time.perf_counter() - start_predict
    convergence_warnings = [w for w in caught if issubclass(w.category, ConvergenceWarning)]
    max_iter_reached = bool((model.n_iter_ >= max_iter).any())
    fit_status = "converged" if not convergence_warnings and not max_iter_reached else "convergence_warning"
    return pred, fit_seconds, predict_seconds, int(len(y_train)), fit_status


def fit_predict_lightgbm(dataset, seed):
    lgb = ensure_lightgbm()
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    model = lgb.LGBMClassifier(
        **LGBM_PARAMS,
        class_weight="balanced",
        random_state=seed,
        verbosity=-1,
    )
    start_fit = time.perf_counter()
    model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    pred = model.predict(dataset["x_validation"])
    predict_seconds = time.perf_counter() - start_predict
    return pred, fit_seconds, predict_seconds, int(len(y_train)), "not_applicable"


def set_global_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch = ensure_torch()
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return torch


def make_simple_gru(input_dim, seed):
    torch = set_global_seed(seed)
    nn = torch.nn

    class SimpleGRUClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.gru = nn.GRU(input_dim, 32, num_layers=1, batch_first=True)
            self.dropout = nn.Dropout(TORCH_DROPOUT)
            self.head = nn.Linear(32, 2)

        def forward(self, x):
            output, _ = self.gru(x)
            return self.head(self.dropout(output[:, -1, :]))

    return SimpleGRUClassifier()


def make_ms_dlinear_tcn(input_dim, window_size, seed):
    torch = set_global_seed(seed)
    nn = torch.nn
    functional = torch.nn.functional

    class CausalConvBlock(nn.Module):
        def __init__(self, in_channels, out_channels, kernel_size, dropout):
            super().__init__()
            self.pad = kernel_size - 1
            self.conv = nn.Conv1d(in_channels, out_channels, kernel_size)
            self.norm = nn.BatchNorm1d(out_channels)
            self.dropout = nn.Dropout(dropout)
            self.proj = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

        def forward(self, x):
            residual = self.proj(x)
            padded = functional.pad(x, (self.pad, 0))
            out = self.conv(padded)
            out = self.dropout(torch.relu(self.norm(out)))
            return out + residual

    class MultiScaleDLinearTCNClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.tcn = nn.Sequential(
                CausalConvBlock(input_dim, TORCH_TCN_CHANNELS[0], TORCH_TCN_KERNEL_SIZE, TORCH_DROPOUT),
                CausalConvBlock(TORCH_TCN_CHANNELS[0], TORCH_TCN_CHANNELS[1], TORCH_TCN_KERNEL_SIZE, TORCH_DROPOUT),
            )
            self.scale_head = nn.Linear(input_dim * len(TORCH_MOVING_AVG_KERNELS), 16)
            self.head = nn.Linear(TORCH_TCN_CHANNELS[-1] + 16, 2)

        def moving_average_last(self, x, kernel):
            pad = kernel - 1
            padded = functional.pad(x.transpose(1, 2), (pad, 0), mode="replicate")
            avg = functional.avg_pool1d(padded, kernel_size=kernel, stride=1)
            return avg[:, :, -1]

        def forward(self, x):
            tcn_last = self.tcn(x.transpose(1, 2))[:, :, -1]
            scale_parts = [self.moving_average_last(x, kernel) for kernel in TORCH_MOVING_AVG_KERNELS]
            scale = torch.relu(self.scale_head(torch.cat(scale_parts, dim=1)))
            return self.head(torch.cat([tcn_last, scale], dim=1))

    return MultiScaleDLinearTCNClassifier()


def run_torch_shape_smoke(input_dim, window_size):
    torch = ensure_torch()
    for model_name, factory in (
        ("simple_gru", lambda: make_simple_gru(input_dim, 41)),
        ("ms_dlinear_tcn", lambda: make_ms_dlinear_tcn(input_dim, window_size, 41)),
    ):
        model = factory()
        x = torch.zeros((2, window_size, input_dim), dtype=torch.float32)
        y = torch.tensor([0, 1], dtype=torch.long)
        logits = model(x)
        if tuple(logits.shape) != (2, 2):
            raise ValueError(f"{model_name} shape smoke failed: logits shape {tuple(logits.shape)}")
        loss = torch.nn.CrossEntropyLoss()(logits, y)
        loss.backward()
    print("Deep adapter shape smoke passed.")


def fit_predict_torch_sequence(dataset, seed, model_name):
    torch = set_global_seed(seed)
    x_train, y_train, train_owner = subsample_rows_with_owner(
        dataset["x_train_seq"],
        dataset["y_train_seq"],
        dataset["train_owner_seq"],
        MAX_TRAIN_ROWS,
        seed=seed,
    )
    x_validation = dataset["x_validation_seq"]
    input_dim = x_train.shape[-1]
    window_size = x_train.shape[1]
    if model_name == "simple_gru":
        model = make_simple_gru(input_dim, seed)
    elif model_name == "ms_dlinear_tcn":
        model = make_ms_dlinear_tcn(input_dim, window_size, seed)
    else:
        raise ValueError(f"Unknown torch model: {model_name}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    train_x_tensor = torch.tensor(x_train, dtype=torch.float32)
    train_y_tensor = torch.tensor(y_train, dtype=torch.long)
    counts = np.bincount(y_train, minlength=2).astype(float)
    class_weights = counts.sum() / np.maximum(counts, 1.0)
    class_weights = class_weights / class_weights.mean()
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=TORCH_LEARNING_RATE, weight_decay=TORCH_WEIGHT_DECAY)
    generator = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(train_x_tensor, train_y_tensor),
        batch_size=TORCH_BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )

    start_fit = time.perf_counter()
    model.train()
    for _ in range(TORCH_EPOCHS):
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
    fit_seconds = time.perf_counter() - start_fit

    start_predict = time.perf_counter()
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(x_validation), TORCH_BATCH_SIZE):
            batch = torch.tensor(x_validation[start : start + TORCH_BATCH_SIZE], dtype=torch.float32, device=device)
            preds.append(model(batch).argmax(dim=1).cpu().numpy())
    predict_seconds = time.perf_counter() - start_predict
    return np.concatenate(preds), fit_seconds, predict_seconds, int(len(y_train)), "fixed_epochs_no_early_stopping"


def fit_predict_model(dataset, model_name, seed):
    if model_name == "logreg":
        return fit_predict_logreg(dataset, seed)
    if model_name == "lightgbm":
        return fit_predict_lightgbm(dataset, seed)
    if model_name in {"simple_gru", "ms_dlinear_tcn"}:
        return fit_predict_torch_sequence(dataset, seed, model_name)
    raise ValueError(f"Unknown model: {model_name}")


def concentration_from_per_ticker(per_ticker_rows):
    deltas = [row["per_ticker_delta_macro_f1_vs_dummy"] for row in per_ticker_rows]
    positive = [float(delta) for delta in deltas if pd.notna(delta) and delta > 0]
    positive_ticker_count = int(len(positive))
    top_ticker_gain_share = float(max(positive) / sum(positive)) if positive else 0.0
    return positive_ticker_count, top_ticker_gain_share


def run_one_model_seed(stage, model_name, label_config, feature_set, window_size, seed):
    dataset = get_dataset(label_config, feature_set, window_size)
    prep_seconds = float(dataset["prep_seconds"])
    pred, fit_seconds, predict_seconds, train_n, fit_status = fit_predict_model(dataset, model_name, seed)
    pooled_metrics = evaluate_predictions(dataset["y_validation"], pred)
    pooled_dummy = dummy_metrics(dataset["y_train"], dataset["y_validation"], seed)
    per_ticker_rows = []
    for ticker in TICKERS:
        val_mask = dataset["validation_owner"] == ticker
        train_mask = dataset["train_owner"] == ticker
        if not val_mask.any():
            continue
        ticker_metrics = evaluate_predictions(dataset["y_validation"][val_mask], pred[val_mask])
        ticker_dummy = dummy_metrics(dataset["y_train"][train_mask], dataset["y_validation"][val_mask], seed)
        per_ticker_rows.append({
            "stage": stage,
            "model": model_name,
            "label_config": label_config,
            "horizon_k": dataset["horizon_k"],
            "threshold_bps": dataset["threshold_bps"],
            "feature_set": feature_set,
            "window_size": int(window_size),
            "seed": int(seed),
            "scope": RESULT_SCOPE,
            "macro_f1": ticker_metrics["macro_f1"],
            "balanced_accuracy": ticker_metrics["balanced_accuracy"],
            "accuracy": ticker_metrics["accuracy"],
            "dummy_macro_f1": ticker_dummy["dummy_macro_f1"],
            "dummy_balanced_accuracy": ticker_dummy["dummy_balanced_accuracy"],
            "delta_macro_f1_vs_dummy": ticker_metrics["macro_f1"] - ticker_dummy["dummy_macro_f1"],
            "delta_balanced_accuracy_vs_dummy": (
                ticker_metrics["balanced_accuracy"] - ticker_dummy["dummy_balanced_accuracy"]
            ),
            "n": int(val_mask.sum()),
            "ticker_or_pooled": ticker,
            "prep_seconds": prep_seconds,
            "fit_seconds": float(fit_seconds),
            "predict_seconds": float(predict_seconds),
            "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
            "per_ticker_delta_macro_f1_vs_dummy": ticker_metrics["macro_f1"] - ticker_dummy["dummy_macro_f1"],
            "positive_ticker_count": np.nan,
            "top_ticker_gain_share": np.nan,
            "train_n": int(train_mask.sum()),
            "fit_status": fit_status,
        })
    positive_ticker_count, top_ticker_gain_share = concentration_from_per_ticker(per_ticker_rows)
    for row in per_ticker_rows:
        row["positive_ticker_count"] = positive_ticker_count
        row["top_ticker_gain_share"] = top_ticker_gain_share
    pooled_row = {
        "stage": stage,
        "model": model_name,
        "label_config": label_config,
        "horizon_k": dataset["horizon_k"],
        "threshold_bps": dataset["threshold_bps"],
        "feature_set": feature_set,
        "window_size": int(window_size),
        "seed": int(seed),
        "scope": RESULT_SCOPE,
        "macro_f1": pooled_metrics["macro_f1"],
        "balanced_accuracy": pooled_metrics["balanced_accuracy"],
        "accuracy": pooled_metrics["accuracy"],
        "dummy_macro_f1": pooled_dummy["dummy_macro_f1"],
        "dummy_balanced_accuracy": pooled_dummy["dummy_balanced_accuracy"],
        "delta_macro_f1_vs_dummy": pooled_metrics["macro_f1"] - pooled_dummy["dummy_macro_f1"],
        "delta_balanced_accuracy_vs_dummy": pooled_metrics["balanced_accuracy"] - pooled_dummy["dummy_balanced_accuracy"],
        "n": int(len(dataset["y_validation"])),
        "ticker_or_pooled": "pooled",
        "prep_seconds": prep_seconds,
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
        "per_ticker_delta_macro_f1_vs_dummy": np.nan,
        "positive_ticker_count": positive_ticker_count,
        "top_ticker_gain_share": top_ticker_gain_share,
        "train_n": int(train_n),
        "fit_status": fit_status,
    }
    return pooled_row, per_ticker_rows


def run_stage_grid(stage, specs):
    pooled_rows = []
    per_ticker_rows = []
    for spec in specs:
        print(
            stage,
            spec["model"],
            spec["label_config"],
            spec["feature_set"],
            "window",
            spec["window_size"],
            "seed",
            spec["seed"],
        )
        pooled, per_ticker = run_one_model_seed(
            stage=stage,
            model_name=spec["model"],
            label_config=spec["label_config"],
            feature_set=spec["feature_set"],
            window_size=spec["window_size"],
            seed=spec["seed"],
        )
        pooled_rows.append(pooled)
        per_ticker_rows.extend(per_ticker)
    return pd.DataFrame(pooled_rows), pd.DataFrame(per_ticker_rows)


def summarize_pooled(pooled):
    if pooled.empty:
        return pd.DataFrame()
    rows = []
    keys = ["stage", "model", "label_config", "horizon_k", "threshold_bps", "feature_set", "window_size", "scope"]
    for key_values, group in pooled.groupby(keys, sort=False):
        record = dict(zip(keys, key_values))
        seed_count = int(group["seed"].nunique())
        macro_std = sample_std(group["macro_f1"])
        bal_std = sample_std(group["balanced_accuracy"])
        record.update({
            "seed_count": seed_count,
            "macro_f1_mean": float(group["macro_f1"].mean()),
            "macro_f1_std": macro_std,
            "macro_f1_lcb_95": float(
                group["macro_f1"].mean()
                - t_critical_one_sided_95(seed_count) * macro_std / math.sqrt(max(seed_count, 1))
            ),
            "balanced_accuracy_mean": float(group["balanced_accuracy"].mean()),
            "balanced_accuracy_std": bal_std,
            "dummy_macro_f1_mean": float(group["dummy_macro_f1"].mean()),
            "dummy_balanced_accuracy_mean": float(group["dummy_balanced_accuracy"].mean()),
            "delta_macro_f1_vs_dummy_mean": float(group["delta_macro_f1_vs_dummy"].mean()),
            "delta_balanced_accuracy_vs_dummy_mean": float(group["delta_balanced_accuracy_vs_dummy"].mean()),
            "n_mean": float(group["n"].mean()),
            "positive_ticker_count": int(round(group["positive_ticker_count"].mean())),
            "top_ticker_gain_share": float(group["top_ticker_gain_share"].mean()),
            "prep_seconds_mean": float(group["prep_seconds"].mean()),
            "fit_seconds_mean": float(group["fit_seconds"].mean()),
            "predict_seconds_mean": float(group["predict_seconds"].mean()),
            "total_seconds_mean": float(group["total_seconds"].mean()),
        })
        record["basic_gate"] = bool(
            record["delta_macro_f1_vs_dummy_mean"] > 0
            and record["macro_f1_lcb_95"] > record["dummy_macro_f1_mean"]
        )
        record["lcb_eligible"] = bool(
            record["basic_gate"]
            and record["delta_balanced_accuracy_vs_dummy_mean"] > 0
            and record["top_ticker_gain_share"] < 0.50
            and record["positive_ticker_count"] >= 3
        )
        rows.append(record)
    return pd.DataFrame(rows)


def tuple_from_row(row, include_window):
    if row is None:
        return None
    if include_window:
        return {
            "label_config": row["label_config"],
            "feature_set": row["feature_set"],
            "window_size": int(row["window_size"]),
        }
    return {"label_config": row["label_config"], "feature_set": row["feature_set"]}


def select_candidates(summary, include_window):
    if summary.empty or not summary["basic_gate"].any():
        return {
            "stage0_result": "do_not_decide_config",
            "mean_candidate": None,
            "lcb_candidate": None,
            "candidate_count": 0,
        }
    basic = summary.loc[summary["basic_gate"]].sort_values("macro_f1_mean", ascending=False)
    lcb = summary.loc[summary["lcb_eligible"]].sort_values("macro_f1_lcb_95", ascending=False)
    mean_candidate = tuple_from_row(basic.iloc[0], include_window=include_window)
    lcb_candidate = tuple_from_row(lcb.iloc[0], include_window=include_window) if not lcb.empty else None
    unique_candidates = []
    for candidate in (mean_candidate, lcb_candidate):
        if candidate is not None and candidate not in unique_candidates:
            unique_candidates.append(candidate)
    return {
        "stage0_result": "candidate_config_selected",
        "mean_candidate": mean_candidate,
        "lcb_candidate": lcb_candidate,
        "candidate_count": len(unique_candidates),
        "candidates": unique_candidates,
    }


def write_outputs(pooled, per_ticker, summary, file_keys):
    pooled.to_csv(OUTPUT_FILES[file_keys[0]], index=False)
    per_ticker.to_csv(OUTPUT_FILES[file_keys[1]], index=False)
    if summary is not None:
        summary.to_csv(OUTPUT_FILES[file_keys[2]], index=False)
    print("wrote", [str(OUTPUT_FILES[key]) for key in file_keys])


## Notebook 04 Controlled Follow-Up Helpers

This layer adds fresh-seed confirmation, prediction artifact persistence, selective coverage, context checks, manual decision matrix, and bootstrap diagnostics.

In [ ]:
\
def assert_sample_alignment(y_flat, y_seq, owner_flat, owner_seq, timestamp_flat, timestamp_seq, split_name):
    if len(y_flat) != len(y_seq):
        raise ValueError(f"{split_name} tabular and sequence labels have different lengths.")
    if not np.array_equal(y_flat, y_seq):
        raise ValueError(f"{split_name} tabular and sequence labels are not aligned.")
    if not np.array_equal(owner_flat, owner_seq):
        raise ValueError(f"{split_name} tabular and sequence ticker owners are not aligned.")
    if not np.array_equal(timestamp_flat, timestamp_seq):
        raise ValueError(f"{split_name} tabular and sequence timestamps are not aligned.")


def make_validation_sample_ids(tickers, timestamps):
    ids = []
    for ticker, timestamp in zip(tickers, timestamps):
        ids.append(f"{ticker}__{pd.Timestamp(timestamp).isoformat()}")
    return np.asarray(ids, dtype=object)


# This get_dataset intentionally overrides the Stage 0 copied helper above.
# It preserves validation timestamps and stable sample ids for prediction
# artifacts and selective-coverage same-row checks.
def get_dataset(label_config, feature_set, window_size):
    key = (label_config, feature_set, int(window_size))
    if key in DATASET_CACHE:
        dataset = DATASET_CACHE[key].copy()
        dataset["prep_seconds"] = 0.0
        return dataset
    if not raw_data:
        raise RuntimeError("raw_data is empty. Enable a Notebook 04 data-loading run switch and rerun setup first.")
    horizon_k, threshold_bps = label_spec(label_config)
    feature_columns = FEATURE_SETS[feature_set]
    start = time.perf_counter()
    split_frames = prepare_split_frames(raw_data, horizon_k=horizon_k, threshold_bps=threshold_bps)
    scaled_frames = fit_transform_train_validation(split_frames, feature_columns)
    x_train, y_train, train_owner, train_timestamp = build_last_step_windows(
        scaled_frames, feature_columns, "train", window_size
    )
    x_validation, y_validation, validation_owner, validation_timestamp = build_last_step_windows(
        scaled_frames, feature_columns, "validation", window_size
    )
    x_train_seq, y_train_seq, train_owner_seq, train_timestamp_seq = build_sequence_windows(
        scaled_frames, feature_columns, "train", window_size
    )
    x_validation_seq, y_validation_seq, validation_owner_seq, validation_timestamp_seq = build_sequence_windows(
        scaled_frames, feature_columns, "validation", window_size
    )
    if len(y_train) == 0 or len(y_validation) == 0:
        raise ValueError(f"No windows available for {label_config} / {feature_set} / window={window_size}")
    assert_sample_alignment(
        y_train,
        y_train_seq,
        train_owner,
        train_owner_seq,
        train_timestamp,
        train_timestamp_seq,
        split_name="train",
    )
    assert_sample_alignment(
        y_validation,
        y_validation_seq,
        validation_owner,
        validation_owner_seq,
        validation_timestamp,
        validation_timestamp_seq,
        split_name="validation",
    )
    validation_sample_id = make_validation_sample_ids(validation_owner, validation_timestamp)
    if len(np.unique(validation_sample_id)) != len(validation_sample_id):
        raise ValueError("Validation sample ids are not unique; inspect ticker/timestamp alignment.")
    dataset = {
        "label_config": label_config,
        "horizon_k": horizon_k,
        "threshold_bps": threshold_bps,
        "feature_set": feature_set,
        "feature_columns": feature_columns,
        "window_size": int(window_size),
        "x_train": x_train,
        "y_train": y_train,
        "train_owner": train_owner,
        "train_timestamp": train_timestamp,
        "x_validation": x_validation,
        "y_validation": y_validation,
        "validation_owner": validation_owner,
        "validation_timestamp": validation_timestamp,
        "validation_sample_id": validation_sample_id,
        "x_train_seq": x_train_seq,
        "y_train_seq": y_train_seq,
        "train_owner_seq": train_owner_seq,
        "train_timestamp_seq": train_timestamp_seq,
        "x_validation_seq": x_validation_seq,
        "y_validation_seq": y_validation_seq,
        "validation_owner_seq": validation_owner_seq,
        "validation_timestamp_seq": validation_timestamp_seq,
        "prep_seconds": time.perf_counter() - start,
    }
    DATASET_CACHE[key] = dataset.copy()
    return dataset


def prediction_diagnostics(y_true, y_pred):
    metrics = evaluate_predictions(y_true, y_pred)
    y_pred = np.asarray(y_pred).astype(int)
    metrics["pred_up_pct"] = float((y_pred == 1).mean()) if len(y_pred) else np.nan
    metrics["pred_down_pct"] = float((y_pred == 0).mean()) if len(y_pred) else np.nan
    return metrics


def stratified_dummy_prediction_payload(dataset, seed):
    start = time.perf_counter()
    dummy = DummyClassifier(strategy="stratified", random_state=seed)
    dummy.fit(np.zeros((len(dataset["y_train"]), 1)), dataset["y_train"])
    y_pred = dummy.predict(np.zeros((len(dataset["y_validation"]), 1))).astype(int)
    probability = dummy.predict_proba(np.zeros((len(dataset["y_validation"]), 1)))
    class_index = list(dummy.classes_).index(1) if 1 in dummy.classes_ else None
    prob_up = probability[:, class_index] if class_index is not None else np.zeros(len(y_pred), dtype=float)
    return y_pred, prob_up.astype(float), 0.0, time.perf_counter() - start, len(dataset["y_train"]), "baseline_stratified"


def always_up_prediction_payload(dataset, seed):
    start = time.perf_counter()
    y_pred = np.ones(len(dataset["y_validation"]), dtype=int)
    prob_up = np.ones(len(dataset["y_validation"]), dtype=float)
    return y_pred, prob_up, 0.0, time.perf_counter() - start, len(dataset["y_train"]), "baseline_always_up"


def fit_predict_logreg_04(dataset, seed):
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    model = LogisticRegression(
        solver="liblinear",
        max_iter=2000,
        class_weight="balanced",
        C=1.0,
        random_state=seed,
    )
    start_fit = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    y_pred = model.predict(dataset["x_validation"]).astype(int)
    prob_up = model.predict_proba(dataset["x_validation"])[:, 1].astype(float)
    predict_seconds = time.perf_counter() - start_predict
    convergence_warnings = [w for w in caught if issubclass(w.category, ConvergenceWarning)]
    max_iter_reached = bool((model.n_iter_ >= 2000).any())
    fit_status = "converged" if not convergence_warnings and not max_iter_reached else "convergence_warning"
    return y_pred, prob_up, fit_seconds, predict_seconds, len(y_train), fit_status


def fit_predict_lightgbm_04(dataset, seed):
    lgb = ensure_lightgbm()
    x_train, y_train = subsample_rows_uniformly(dataset["x_train"], dataset["y_train"], MAX_TRAIN_ROWS, seed=seed)
    model = lgb.LGBMClassifier(
        **LGBM_PARAMS,
        class_weight="balanced",
        random_state=seed,
        verbosity=-1,
    )
    start_fit = time.perf_counter()
    model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start_fit
    start_predict = time.perf_counter()
    y_pred = model.predict(dataset["x_validation"]).astype(int)
    prob_up = model.predict_proba(dataset["x_validation"])[:, 1].astype(float)
    predict_seconds = time.perf_counter() - start_predict
    return y_pred, prob_up, fit_seconds, predict_seconds, len(y_train), "not_applicable"


def make_standalone_tcn(input_dim, seed):
    torch = set_global_seed(seed)
    nn = torch.nn

    class CausalConvBlock(nn.Module):
        def __init__(self, channels_in, channels_out, kernel_size, dropout):
            super().__init__()
            self.padding = kernel_size - 1
            self.conv = nn.Conv1d(channels_in, channels_out, kernel_size, padding=self.padding)
            self.activation = nn.ReLU()
            self.dropout = nn.Dropout(dropout)
            self.residual = nn.Conv1d(channels_in, channels_out, 1) if channels_in != channels_out else nn.Identity()

        def forward(self, x):
            conv_out = self.conv(x)
            if self.padding:
                conv_out = conv_out[:, :, :-self.padding]
            return self.dropout(self.activation(conv_out)) + self.residual(x)

    class StandaloneTCNClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            channels = (input_dim,) + tuple(TORCH_TCN_CHANNELS)
            self.blocks = nn.Sequential(*[
                CausalConvBlock(channels[idx], channels[idx + 1], TORCH_TCN_KERNEL_SIZE, TORCH_DROPOUT)
                for idx in range(len(channels) - 1)
            ])
            self.head = nn.Linear(channels[-1], 2)

        def forward(self, x):
            z = x.transpose(1, 2)
            z = self.blocks(z)
            return self.head(z[:, :, -1])

    return StandaloneTCNClassifier()


def make_sequence_model_04(model_name, input_dim, window_size, seed):
    if model_name == "standalone_tcn":
        return make_standalone_tcn(input_dim, seed)
    if model_name == "ms_dlinear_tcn":
        return make_ms_dlinear_tcn(input_dim, window_size, seed)
    raise ValueError(f"Unsupported Notebook 04 sequence model: {model_name}")


def fit_predict_torch_sequence_04(dataset, seed, model_name):
    torch = set_global_seed(seed)
    x_train, y_train, train_owner = subsample_rows_with_owner(
        dataset["x_train_seq"],
        dataset["y_train_seq"],
        dataset["train_owner_seq"],
        MAX_TRAIN_ROWS,
        seed=seed,
    )
    x_validation = dataset["x_validation_seq"]
    input_dim = x_train.shape[-1]
    window_size = x_train.shape[1]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = make_sequence_model_04(model_name, input_dim, window_size, seed)
    model.to(device)
    train_x_tensor = torch.tensor(x_train, dtype=torch.float32)
    train_y_tensor = torch.tensor(y_train, dtype=torch.long)
    counts = np.bincount(y_train.astype(int), minlength=2).astype(float)
    class_weights = counts.sum() / np.maximum(counts, 1.0)
    class_weights = class_weights / class_weights.mean()
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=TORCH_LEARNING_RATE, weight_decay=TORCH_WEIGHT_DECAY)
    generator = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(train_x_tensor, train_y_tensor),
        batch_size=TORCH_BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )

    start_fit = time.perf_counter()
    model.train()
    for _ in range(TORCH_EPOCHS):
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            if not torch.isfinite(loss):
                raise ValueError(f"{model_name} loss became non-finite.")
            loss.backward()
            optimizer.step()
    fit_seconds = time.perf_counter() - start_fit

    start_predict = time.perf_counter()
    model.eval()
    prob_parts = []
    with torch.no_grad():
        for start in range(0, len(x_validation), TORCH_BATCH_SIZE):
            batch = torch.tensor(x_validation[start : start + TORCH_BATCH_SIZE], dtype=torch.float32, device=device)
            logits = model(batch)
            prob_parts.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    prob_up = np.concatenate(prob_parts).astype(float)
    y_pred = (prob_up >= 0.5).astype(int)
    predict_seconds = time.perf_counter() - start_predict
    return y_pred, prob_up, fit_seconds, predict_seconds, len(y_train), f"fixed_epochs_device_{device}"


def fit_predict_model_04(dataset, model_name, seed):
    if model_name == "stratified_dummy":
        return stratified_dummy_prediction_payload(dataset, seed)
    if model_name == "always_up_dummy":
        return always_up_prediction_payload(dataset, seed)
    if model_name == "logreg":
        return fit_predict_logreg_04(dataset, seed)
    if model_name == "lightgbm":
        return fit_predict_lightgbm_04(dataset, seed)
    if model_name in SEQUENCE_MODELS:
        return fit_predict_torch_sequence_04(dataset, seed, model_name)
    raise ValueError(f"Unknown Notebook 04 model: {model_name}")


def run_notebook04_shape_smoke():
    synthetic_sample_id = np.asarray(["CSCO__synthetic_0", "JPM__synthetic_1"], dtype=object)
    synthetic_ticker = np.asarray(["CSCO", "JPM"], dtype=object)
    synthetic_timestamp = np.asarray(["2026-06-04T09:30:00", "2026-06-04T09:35:00"], dtype=object)
    synthetic_y_true = np.asarray([0, 1], dtype=int)
    synthetic_y_pred = np.asarray([0, 1], dtype=int)
    synthetic_prob_up = np.asarray([0.25, 0.75], dtype=float)
    artifact = {
        "validation_sample_id": synthetic_sample_id,
        "ticker": synthetic_ticker,
        "timestamp": synthetic_timestamp,
        "y_true": synthetic_y_true,
        "y_pred": synthetic_y_pred,
        "prob_up": synthetic_prob_up,
        "confidence": np.maximum(synthetic_prob_up, 1.0 - synthetic_prob_up),
    }
    validate_prediction_payload(artifact)
    torch = ensure_torch()
    for model_name in SEQUENCE_MODELS:
        model = make_sequence_model_04(model_name, input_dim=3, window_size=20, seed=606)
        model.eval()
        x = torch.zeros((2, 20, 3), dtype=torch.float32)
        with torch.no_grad():
            logits = model(x)
        if tuple(logits.shape) != (2, 2):
            raise ValueError(f"{model_name} smoke output shape mismatch: {tuple(logits.shape)}")
    print("Notebook 04 schema and torch shape smoke passed.")


def validate_prediction_payload(payload):
    required = ("validation_sample_id", "ticker", "timestamp", "y_true", "y_pred", "prob_up", "confidence")
    lengths = []
    for name in required:
        if name not in payload:
            raise ValueError(f"Prediction payload missing array: {name}")
        lengths.append(len(payload[name]))
    if len(set(lengths)) != 1:
        raise ValueError(f"Prediction payload lengths disagree: {dict(zip(required, lengths))}")
    prob_up = np.asarray(payload["prob_up"], dtype=float)
    confidence = np.asarray(payload["confidence"], dtype=float)
    if np.isnan(prob_up).any() or np.isnan(confidence).any():
        raise ValueError("Prediction payload contains NaN probability or confidence values.")
    if ((prob_up < 0.0) | (prob_up > 1.0)).any():
        raise ValueError("prob_up values must be within [0.0, 1.0].")
    if ((confidence < 0.5) | (confidence > 1.0)).any():
        raise ValueError("confidence values must be within [0.5, 1.0].")


def save_prediction_artifact(dataset, model_name, seed, y_pred, prob_up):
    prob_up = np.asarray(prob_up, dtype=float)
    confidence = np.maximum(prob_up, 1.0 - prob_up)
    payload = {
        "validation_sample_id": dataset["validation_sample_id"],
        "ticker": dataset["validation_owner"].astype(object),
        "timestamp": dataset["validation_timestamp"].astype("datetime64[ns]").astype(str).astype(object),
        "y_true": dataset["y_validation"].astype(int),
        "y_pred": np.asarray(y_pred, dtype=int),
        "prob_up": prob_up,
        "confidence": confidence,
    }
    validate_prediction_payload(payload)
    artifact_path = PREDICTION_DIR / f"{model_name}__seed{int(seed)}.npz"
    np.savez_compressed(artifact_path, **payload)
    row = {
        "model": model_name,
        "seed": int(seed),
        "path": str(artifact_path),
        "row_count": int(len(payload["y_true"])),
        "selective_eligible": bool(model_name in REAL_MODELS),
        "prob_up_source": "predict_proba[:, 1]" if model_name in TABULAR_MODELS else (
            "softmax(logits)[:, 1]" if model_name in SEQUENCE_MODELS else "baseline_probability"
        ),
        "scope": RESULT_SCOPE,
    }
    NOTEBOOK04_STATE["prediction_manifest_rows"].append(row)
    return artifact_path, row


def stratified_dummy_predictions_same_rows(y_train, retained_n, seed):
    dummy = DummyClassifier(strategy="stratified", random_state=seed)
    dummy.fit(np.zeros((len(y_train), 1)), y_train)
    return dummy.predict(np.zeros((retained_n, 1))).astype(int)


def concentration_from_per_ticker(per_ticker_rows):
    deltas = [row["delta_macro_f1_vs_stratified_dummy"] for row in per_ticker_rows]
    positive = [float(delta) for delta in deltas if pd.notna(delta) and delta > 0]
    positive_ticker_count = int(len(positive))
    top_ticker_gain_share = float(max(positive) / sum(positive)) if positive else 0.0
    return positive_ticker_count, top_ticker_gain_share


def row_metrics(y_true, y_pred, stratified_pred, always_up_pred):
    metrics = prediction_diagnostics(y_true, y_pred)
    stratified_metrics = evaluate_predictions(y_true, stratified_pred)
    always_up_metrics = evaluate_predictions(y_true, always_up_pred)
    metrics.update({
        "stratified_dummy_macro_f1": stratified_metrics["macro_f1"],
        "stratified_dummy_balanced_accuracy": stratified_metrics["balanced_accuracy"],
        "delta_macro_f1_vs_stratified_dummy": metrics["macro_f1"] - stratified_metrics["macro_f1"],
        "delta_balanced_accuracy_vs_stratified_dummy": metrics["balanced_accuracy"] - stratified_metrics["balanced_accuracy"],
        "always_up_dummy_macro_f1": always_up_metrics["macro_f1"],
        "always_up_dummy_balanced_accuracy": always_up_metrics["balanced_accuracy"],
        "delta_macro_f1_vs_always_up_dummy": metrics["macro_f1"] - always_up_metrics["macro_f1"],
        "delta_balanced_accuracy_vs_always_up_dummy": metrics["balanced_accuracy"] - always_up_metrics["balanced_accuracy"],
    })
    return metrics


def run_one_notebook04_seed(model_name, seed):
    dataset = get_dataset(
        NOTEBOOK04_CANDIDATE["label_config"],
        NOTEBOOK04_CANDIDATE["feature_set"],
        NOTEBOOK04_CANDIDATE["window_size"],
    )
    prep_seconds = float(dataset.get("prep_seconds", 0.0))
    try:
        y_pred, prob_up, fit_seconds, predict_seconds, train_n, fit_status = fit_predict_model_04(dataset, model_name, seed)
        artifact_path, _ = save_prediction_artifact(dataset, model_name, seed, y_pred, prob_up)
        stratified_pred, _, _, _, _, _ = stratified_dummy_prediction_payload(dataset, seed)
        always_up_pred, _, _, _, _, _ = always_up_prediction_payload(dataset, seed)
        pooled_metrics = row_metrics(dataset["y_validation"], y_pred, stratified_pred, always_up_pred)
        per_ticker_rows = []
        for ticker in TICKERS:
            val_mask = dataset["validation_owner"] == ticker
            if not val_mask.any():
                continue
            ticker_metrics = row_metrics(
                dataset["y_validation"][val_mask],
                np.asarray(y_pred)[val_mask],
                stratified_pred[val_mask],
                always_up_pred[val_mask],
            )
            per_ticker_rows.append({
                "stage": "04B_fresh_seed_panel",
                "candidate_id": NOTEBOOK04_CANDIDATE["candidate_id"],
                "model": model_name,
                "seed": int(seed),
                "label_config": dataset["label_config"],
                "horizon_k": dataset["horizon_k"],
                "threshold_bps": dataset["threshold_bps"],
                "feature_set": dataset["feature_set"],
                "window_size": int(dataset["window_size"]),
                "scope": RESULT_SCOPE,
                "ticker_or_pooled": ticker,
                "n": int(val_mask.sum()),
                "train_n": int((dataset["train_owner"] == ticker).sum()),
                "validation_n": int(val_mask.sum()),
                "run_failed": False,
                "failure_reason": "",
                "fit_status": fit_status,
                "fit_seconds": float(fit_seconds),
                "predict_seconds": float(predict_seconds),
                "prep_seconds": prep_seconds,
                "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
                "prediction_artifact": str(artifact_path),
                **ticker_metrics,
            })
        positive_ticker_count, top_ticker_gain_share = concentration_from_per_ticker(per_ticker_rows)
        for row in per_ticker_rows:
            row["positive_ticker_count"] = positive_ticker_count
            row["top_ticker_gain_share"] = top_ticker_gain_share
        pooled_row = {
            "stage": "04B_fresh_seed_panel",
            "candidate_id": NOTEBOOK04_CANDIDATE["candidate_id"],
            "model": model_name,
            "seed": int(seed),
            "label_config": dataset["label_config"],
            "horizon_k": dataset["horizon_k"],
            "threshold_bps": dataset["threshold_bps"],
            "feature_set": dataset["feature_set"],
            "window_size": int(dataset["window_size"]),
            "scope": RESULT_SCOPE,
            "ticker_or_pooled": "pooled",
            "n": int(len(dataset["y_validation"])),
            "train_n": int(train_n),
            "validation_n": int(len(dataset["y_validation"])),
            "run_failed": False,
            "failure_reason": "",
            "fit_status": fit_status,
            "fit_seconds": float(fit_seconds),
            "predict_seconds": float(predict_seconds),
            "prep_seconds": prep_seconds,
            "total_seconds": prep_seconds + float(fit_seconds) + float(predict_seconds),
            "prediction_artifact": str(artifact_path),
            "positive_ticker_count": positive_ticker_count,
            "top_ticker_gain_share": top_ticker_gain_share,
            **pooled_metrics,
        }
    except Exception as exc:
        pooled_row = {
            "stage": "04B_fresh_seed_panel",
            "candidate_id": NOTEBOOK04_CANDIDATE["candidate_id"],
            "model": model_name,
            "seed": int(seed),
            "label_config": NOTEBOOK04_CANDIDATE["label_config"],
            "horizon_k": LABEL_CONFIGS[NOTEBOOK04_CANDIDATE["label_config"]]["horizon_k"],
            "threshold_bps": LABEL_CONFIGS[NOTEBOOK04_CANDIDATE["label_config"]]["threshold_bps"],
            "feature_set": NOTEBOOK04_CANDIDATE["feature_set"],
            "window_size": int(NOTEBOOK04_CANDIDATE["window_size"]),
            "scope": RESULT_SCOPE,
            "ticker_or_pooled": "pooled",
            "n": 0,
            "train_n": 0,
            "validation_n": 0,
            "run_failed": True,
            "failure_reason": f"{type(exc).__name__}: {exc}",
            "fit_status": "run_failed",
        }
        for column in (
            "macro_f1",
            "balanced_accuracy",
            "accuracy",
            "stratified_dummy_macro_f1",
            "stratified_dummy_balanced_accuracy",
            "delta_macro_f1_vs_stratified_dummy",
            "delta_balanced_accuracy_vs_stratified_dummy",
            "always_up_dummy_macro_f1",
            "always_up_dummy_balanced_accuracy",
            "delta_macro_f1_vs_always_up_dummy",
            "delta_balanced_accuracy_vs_always_up_dummy",
            "pred_up_pct",
            "pred_down_pct",
            "fit_seconds",
            "predict_seconds",
            "prep_seconds",
            "total_seconds",
            "positive_ticker_count",
            "top_ticker_gain_share",
        ):
            pooled_row[column] = np.nan
        per_ticker_rows = []
    return pooled_row, per_ticker_rows


def summarize_notebook04(pooled):
    if pooled.empty:
        return pd.DataFrame()
    rows = []
    keys = ["candidate_id", "model", "label_config", "horizon_k", "threshold_bps", "feature_set", "window_size", "scope"]
    notebook03_summary = read_optional_notebook03_summary()
    for key_values, group in pooled.groupby(keys, sort=False):
        record = dict(zip(keys, key_values))
        successful = group.loc[~group["run_failed"].astype(bool)].copy()
        seed_count = int(successful["seed"].nunique())
        n_failed = int(group["run_failed"].astype(bool).sum())
        record["seed_count"] = seed_count
        record["n_failed_seeds"] = n_failed
        record["run_failed"] = bool(n_failed > 0)
        record["failure_reason"] = "; ".join(sorted(set(group.loc[group["run_failed"].astype(bool), "failure_reason"].dropna().astype(str))))
        if successful.empty:
            for column in (
                "macro_f1_mean",
                "macro_f1_std",
                "macro_f1_lcb_95",
                "balanced_accuracy_mean",
                "stratified_dummy_macro_f1_mean",
                "delta_macro_f1_vs_stratified_dummy_mean",
                "delta_balanced_accuracy_vs_stratified_dummy_mean",
                "always_up_dummy_macro_f1_mean",
                "delta_macro_f1_vs_always_up_dummy_mean",
                "positive_ticker_count",
                "top_ticker_gain_share",
                "fresh_minus_03",
            ):
                record[column] = np.nan
            record["basic_gate_pass"] = False
            record["fresh_seed_stability_tag"] = "run_failed"
        else:
            macro_std = sample_std(successful["macro_f1"])
            macro_mean = float(successful["macro_f1"].mean())
            record.update({
                "macro_f1_mean": macro_mean,
                "macro_f1_std": macro_std,
                "macro_f1_lcb_95": float(
                    macro_mean - t_critical_one_sided_95(seed_count) * macro_std / math.sqrt(max(seed_count, 1))
                ),
                "balanced_accuracy_mean": float(successful["balanced_accuracy"].mean()),
                "stratified_dummy_macro_f1_mean": float(successful["stratified_dummy_macro_f1"].mean()),
                "delta_macro_f1_vs_stratified_dummy_mean": float(successful["delta_macro_f1_vs_stratified_dummy"].mean()),
                "delta_balanced_accuracy_vs_stratified_dummy_mean": float(successful["delta_balanced_accuracy_vs_stratified_dummy"].mean()),
                "always_up_dummy_macro_f1_mean": float(successful["always_up_dummy_macro_f1"].mean()),
                "delta_macro_f1_vs_always_up_dummy_mean": float(successful["delta_macro_f1_vs_always_up_dummy"].mean()),
                "n_mean": float(successful["n"].mean()),
                "positive_ticker_count": int(round(successful["positive_ticker_count"].mean())),
                "top_ticker_gain_share": float(successful["top_ticker_gain_share"].mean()),
            })
            record["basic_gate_pass"] = bool(
                record["model"] in REAL_MODELS
                and n_failed == 0
                and record["macro_f1_lcb_95"] > record["stratified_dummy_macro_f1_mean"]
                and record["delta_macro_f1_vs_stratified_dummy_mean"] > 0
                and record["positive_ticker_count"] >= 3
                and record["top_ticker_gain_share"] <= 0.50
            )
            notebook03_macro = lookup_notebook03_macro(notebook03_summary, record["model"])
            record["notebook03_macro_f1_mean"] = notebook03_macro
            record["fresh_minus_03"] = macro_mean - notebook03_macro if pd.notna(notebook03_macro) else np.nan
            if pd.isna(record["fresh_minus_03"]):
                record["fresh_seed_stability_tag"] = "notebook03_reference_missing"
            elif record["fresh_minus_03"] >= -0.001:
                record["fresh_seed_stability_tag"] = "confirmed_or_improved"
            elif -0.003 < record["fresh_minus_03"] < -0.001:
                record["fresh_seed_stability_tag"] = "marginal_drop_note_only"
            else:
                record["fresh_seed_stability_tag"] = "failed_fresh_seed_confirmation"
            record["positive_shift_review"] = bool(pd.notna(record["fresh_minus_03"]) and record["fresh_minus_03"] > 0.003)
        rows.append(record)
    return pd.DataFrame(rows)


def drive_query_literal(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def find_latest_drive_file_by_suffix(service, folder_id, filename_suffix):
    escaped_parent = drive_query_literal(folder_id)
    query = f"{escaped_parent} in parents and trashed = false"
    response = service.files().list(
        q=query,
        spaces="drive",
        fields="files(id,name,mimeType,createdTime,modifiedTime)",
        pageSize=100,
    ).execute()
    files = [
        item
        for item in response.get("files", [])
        if str(item.get("name", "")).endswith(filename_suffix)
    ]
    if not files:
        raise FileNotFoundError(
            f"No Drive file ending with {filename_suffix!r} found in folder "
            f"{NOTEBOOK03_DRIVE_RESULTS_FOLDER_NAME} ({folder_id})."
        )
    return sorted(files, key=lambda item: str(item.get("name", "")), reverse=True)[0]


def download_drive_file(service, file_id, target_path):
    from googleapiclient.http import MediaIoBaseDownload

    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with target_path.open("wb") as output:
        downloader = MediaIoBaseDownload(output, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return target_path


def ensure_latest_notebook03_context_from_drive():
    selection_target = NOTEBOOK03_SELECTION_CANDIDATES[0]
    summary_target = NOTEBOOK03_SUMMARY_CANDIDATES[0]
    if selection_target.exists() and summary_target.exists():
        return {
            "selection": str(selection_target),
            "summary": str(summary_target),
            "source": "local_existing",
        }

    service = build_drive_service()
    downloaded = {}
    if not selection_target.exists():
        selection_file = find_latest_drive_file_by_suffix(
            service,
            NOTEBOOK03_DRIVE_RESULTS_FOLDER_ID,
            "notebook03_validation_selection.json",
        )
        download_drive_file(service, selection_file["id"], selection_target)
        downloaded["selection"] = {
            "drive_name": selection_file["name"],
            "local_path": str(selection_target),
        }
        print("Downloaded Notebook 03 selection:", selection_file["name"], "->", selection_target)

    if not summary_target.exists():
        summary_file = find_latest_drive_file_by_suffix(
            service,
            NOTEBOOK03_DRIVE_RESULTS_FOLDER_ID,
            "notebook03_summary.csv",
        )
        download_drive_file(service, summary_file["id"], summary_target)
        downloaded["summary"] = {
            "drive_name": summary_file["name"],
            "local_path": str(summary_target),
        }
        print("Downloaded Notebook 03 summary:", summary_file["name"], "->", summary_target)

    return downloaded


def read_optional_notebook03_summary():
    if not any(path.exists() for path in NOTEBOOK03_SUMMARY_CANDIDATES):
        try:
            ensure_latest_notebook03_context_from_drive()
        except Exception as exc:
            print("Notebook 03 summary auto-download skipped:", type(exc).__name__, exc)
    for path in NOTEBOOK03_SUMMARY_CANDIDATES:
        if path.exists():
            return pd.read_csv(path)
    return pd.DataFrame()


def lookup_notebook03_macro(summary, model_name):
    if summary.empty or "model" not in summary.columns or "macro_f1_mean" not in summary.columns:
        return np.nan
    rows = summary.loc[summary["model"].eq(model_name)]
    if rows.empty:
        return np.nan
    return float(rows.iloc[0]["macro_f1_mean"])


def write_run_manifest(pooled, per_ticker, summary, prediction_manifest):
    manifest = {
        "scope": RESULT_SCOPE,
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "candidate": NOTEBOOK04_CANDIDATE,
        "model_panel": list(MODEL_PANEL),
        "fresh_seeds": list(FRESH_SEEDS),
        "row_counts": {
            "pooled": int(len(pooled)),
            "per_ticker": int(len(per_ticker)),
            "summary": int(len(summary)),
            "prediction_manifest": int(len(prediction_manifest)),
        },
        "holdout_test_authorized": False,
        "run_switches": {
            "RUN_04S_SCHEMA_SMOKE": RUN_04S_SCHEMA_SMOKE,
            "RUN_04A_READ_CONTEXT": RUN_04A_READ_CONTEXT,
            "RUN_04B_FRESH_SEED_PANEL": RUN_04B_FRESH_SEED_PANEL,
            "RUN_04C_SELECTIVE_COVERAGE": RUN_04C_SELECTIVE_COVERAGE,
            "RUN_04D_GATE_DECISION": RUN_04D_GATE_DECISION,
            "RUN_04E_BOOTSTRAP_CI": RUN_04E_BOOTSTRAP_CI,
        },
    }
    with OUTPUT_FILES["run_manifest"].open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2)


def find_or_create_drive_folder(service, folder_name, parent_id):
    escaped_name = drive_query_literal(folder_name)
    escaped_parent = drive_query_literal(parent_id)
    query = (
        f"name = {escaped_name} and {escaped_parent} in parents and "
        "mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    )
    response = service.files().list(
        q=query,
        spaces="drive",
        fields="files(id,name,webViewLink)",
        pageSize=10,
    ).execute()
    folders = response.get("files", [])
    if folders:
        return folders[0]
    metadata = {
        "name": folder_name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id],
    }
    return service.files().create(body=metadata, fields="id,name,webViewLink").execute()


def upload_local_file_to_drive(service, local_path, folder_id, uploaded_name):
    import mimetypes
    from googleapiclient.http import MediaFileUpload

    local_path = Path(local_path)
    mime_type = mimetypes.guess_type(str(local_path))[0] or "application/octet-stream"
    media = MediaFileUpload(str(local_path), mimetype=mime_type, resumable=True)
    metadata = {"name": uploaded_name, "parents": [folder_id]}
    return service.files().create(
        body=metadata,
        media_body=media,
        fields="id,name,webViewLink",
    ).execute()


def notebook04_existing_output_paths(include_predictions=False, prediction_model_name=None):
    paths = [path for path in OUTPUT_FILES.values() if path.exists()]
    if include_predictions:
        if prediction_model_name:
            paths.extend(sorted(PREDICTION_DIR.glob(f"{prediction_model_name}__seed*.npz")))
        else:
            paths.extend(sorted(PREDICTION_DIR.glob("*.npz")))
    return paths


def backup_notebook04_outputs(reason, include_predictions=False, prediction_model_name=None):
    if not BACKUP_NOTEBOOK04_TO_GOOGLE_DRIVE:
        return []
    paths = notebook04_existing_output_paths(
        include_predictions=include_predictions,
        prediction_model_name=prediction_model_name,
    )
    if not paths:
        print("Notebook 04 backup skipped; no local output files exist yet.")
        return []

    service = build_drive_service()
    backup_folder = find_or_create_drive_folder(
        service,
        NOTEBOOK04_DRIVE_BACKUP_FOLDER_NAME,
        DRIVE_PROJECT_FOLDER_ID,
    )
    timestamp = pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")
    uploaded = []
    for path in paths:
        uploaded_name = f"{timestamp}__{reason}__{path.name}"
        drive_file = upload_local_file_to_drive(service, path, backup_folder["id"], uploaded_name)
        uploaded.append({
            "local_path": str(path),
            "uploaded_name": uploaded_name,
            "drive_file": drive_file,
        })
        print("Uploaded Notebook 04 artifact:", uploaded_name)

    manifest = {
        "scope": RESULT_SCOPE,
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "reason": reason,
        "include_predictions": bool(include_predictions),
        "prediction_model_name": prediction_model_name,
        "backup_folder": backup_folder,
        "uploaded": uploaded,
        "holdout_test_authorized": False,
    }
    manifest_path = OUTPUT_DIR / f"{timestamp}__{reason}__notebook04_drive_backup_manifest.json"
    with manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2)
    manifest_file = upload_local_file_to_drive(
        service,
        manifest_path,
        backup_folder["id"],
        manifest_path.name,
    )
    uploaded.append({
        "local_path": str(manifest_path),
        "uploaded_name": manifest_path.name,
        "drive_file": manifest_file,
    })
    print("Uploaded Notebook 04 backup manifest:", manifest_path.name)
    return uploaded


def read_first_existing_json(candidates, description):
    missing = []
    for path in candidates:
        if path.exists():
            with path.open("r", encoding="utf-8") as handle:
                return path, json.load(handle)
        missing.append(str(path))
    raise FileNotFoundError(f"Missing {description}. Checked: {'; '.join(missing)}")


def run_context_check_04a():
    drive_context_download = ensure_latest_notebook03_context_from_drive()
    selection_path, selection = read_first_existing_json(NOTEBOOK03_SELECTION_CANDIDATES, "Notebook 03 selection JSON")
    if selection.get("scope") != RESULT_SCOPE:
        raise ValueError(f"Notebook 03 selection scope must be {RESULT_SCOPE}; found {selection.get('scope')!r}.")
    if selection.get("holdout_test_authorized") is not False:
        raise ValueError("Notebook 03 selection must have holdout_test_authorized set to false.")
    candidate_ok = (
        NOTEBOOK04_CANDIDATE["label_config"] == "h03_bps1p5"
        and NOTEBOOK04_CANDIDATE["feature_set"] == "price_volume_time"
        and int(NOTEBOOK04_CANDIDATE["window_size"]) == 20
    )
    if not candidate_ok:
        raise ValueError(f"Notebook 04 official candidate drifted: {NOTEBOOK04_CANDIDATE}")
    h0_status = "not_found"
    for path in H0_SUMMARY_CANDIDATES:
        if path.exists():
            h0 = pd.read_csv(path)
            if "scope" in h0.columns and not h0["scope"].astype(str).str.contains("diagnostic", case=False).all():
                raise ValueError(f"H0 file has non-diagnostic scope values: {path}")
            if "confirmation_status" in h0.columns:
                selected_like = h0["confirmation_status"].astype(str).str.contains("selected", case=False, na=False)
                if selected_like.any() and not h0["confirmation_status"].astype(str).str.contains("not_selected", case=False, na=False).all():
                    raise ValueError(f"H0 file appears to contain selecting confirmation status: {path}")
            h0_status = f"diagnostic_read_only:{path}"
            break
    context = {
        "scope": RESULT_SCOPE,
        "notebook03_selection_json": str(selection_path),
        "notebook03_drive_context_download": drive_context_download,
        "official_candidate": NOTEBOOK04_CANDIDATE,
        "h0_status": h0_status,
        "model_panel": list(MODEL_PANEL),
        "fresh_seeds": list(FRESH_SEEDS),
        "holdout_test_authorized": False,
        "checks_passed": True,
    }
    with OUTPUT_FILES["context"].open("w", encoding="utf-8") as handle:
        json.dump(context, handle, indent=2)
    return context


def write_notebook04_panel_outputs(pooled_rows, per_ticker_rows):
    pooled = pd.DataFrame(pooled_rows)
    per_ticker = pd.DataFrame(per_ticker_rows)
    summary = summarize_notebook04(pooled)
    prediction_manifest = pd.DataFrame(NOTEBOOK04_STATE["prediction_manifest_rows"])
    pooled.to_csv(OUTPUT_FILES["pooled"], index=False)
    per_ticker.to_csv(OUTPUT_FILES["per_ticker"], index=False)
    summary.to_csv(OUTPUT_FILES["summary"], index=False)
    prediction_manifest.to_csv(OUTPUT_FILES["prediction_manifest"], index=False)
    write_run_manifest(pooled, per_ticker, summary, prediction_manifest)
    return pooled, per_ticker, summary, prediction_manifest


def run_fresh_seed_panel_04b():
    pooled_rows = []
    per_ticker_rows = []
    NOTEBOOK04_STATE["prediction_manifest_rows"] = []
    pooled = per_ticker = summary = prediction_manifest = None
    for model_name in MODEL_PANEL:
        for seed in FRESH_SEEDS:
            print("04B", model_name, "seed", seed)
            pooled_row, ticker_rows = run_one_notebook04_seed(model_name, seed)
            pooled_rows.append(pooled_row)
            per_ticker_rows.extend(ticker_rows)
        pooled, per_ticker, summary, prediction_manifest = write_notebook04_panel_outputs(
            pooled_rows,
            per_ticker_rows,
        )
        backup_notebook04_outputs(
            f"checkpoint_04B_after_{model_name}",
            include_predictions=True,
            prediction_model_name=model_name,
        )
    backup_notebook04_outputs("completed_04B_fresh_seed_panel")
    return pooled, per_ticker, summary, prediction_manifest


def load_prediction_npz(path):
    with np.load(path, allow_pickle=True) as loaded:
        payload = {name: loaded[name] for name in loaded.files}
    validate_prediction_payload(payload)
    return payload


def load_prediction_artifact(model_name, seed):
    manifest = pd.read_csv(OUTPUT_FILES["prediction_manifest"])
    rows = manifest.loc[manifest["model"].eq(model_name) & manifest["seed"].astype(int).eq(int(seed))]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one prediction artifact for {model_name} seed {seed}; found {len(rows)}")
    return load_prediction_npz(rows.iloc[0]["path"])


def precision_for_label(y_true, y_pred, label):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    pred_mask = y_pred == int(label)
    if not pred_mask.any():
        return np.nan
    return float((y_true[pred_mask] == int(label)).mean())


def selective_rows_for_artifact(model_name, seed, model_payload, stratified_payload, always_up_payload):
    if not np.array_equal(model_payload["validation_sample_id"], stratified_payload["validation_sample_id"]):
        raise ValueError(f"Stratified dummy sample ids do not match {model_name} seed {seed}.")
    if not np.array_equal(model_payload["validation_sample_id"], always_up_payload["validation_sample_id"]):
        raise ValueError(f"Always-up dummy sample ids do not match {model_name} seed {seed}.")
    order = np.lexsort((model_payload["validation_sample_id"].astype(str), -model_payload["confidence"].astype(float)))
    rows = []
    validation_n = len(order)
    for coverage in SELECTIVE_COVERAGE_GRID:
        retained_n = max(1, int(math.ceil(validation_n * float(coverage))))
        idx = order[:retained_n]
        y_true = model_payload["y_true"][idx].astype(int)
        y_pred = model_payload["y_pred"][idx].astype(int)
        stratified_pred = stratified_payload["y_pred"][idx].astype(int)
        always_up_pred = always_up_payload["y_pred"][idx].astype(int)
        metrics = row_metrics(y_true, y_pred, stratified_pred, always_up_pred)
        retained_ticker = model_payload["ticker"][idx].astype(str)
        ticker_counts = pd.Series(retained_ticker).value_counts()
        class_values = pd.Series(y_true).value_counts()
        pred_values = pd.Series(y_pred).value_counts()
        warnings_list = []
        if coverage <= 0.20:
            warnings_list.append("low_coverage_exploratory")
        if int(ticker_counts.min()) < 500:
            warnings_list.append("per_ticker_retained_n_low")
        if float(ticker_counts.max() / retained_n) > 0.40:
            warnings_list.append("ticker_concentration_warning")
        auc = np.nan
        if len(class_values) == 2:
            try:
                from sklearn.metrics import roc_auc_score
                auc = float(roc_auc_score(y_true, model_payload["prob_up"][idx].astype(float)))
            except ValueError:
                auc = np.nan
        else:
            warnings_list.append("auc_not_defined")
        rows.append({
            "model": model_name,
            "seed": int(seed),
            "coverage": float(coverage),
            "retained_n": int(retained_n),
            "retained_pct": float(retained_n / validation_n),
            "class0_n": int(class_values.get(0, 0)),
            "class1_n": int(class_values.get(1, 0)),
            "pred0_n": int(pred_values.get(0, 0)),
            "pred1_n": int(pred_values.get(1, 0)),
            "selective_error": float(1.0 - metrics["accuracy"]),
            "precision_down": precision_for_label(y_true, y_pred, 0),
            "precision_up": precision_for_label(y_true, y_pred, 1),
            "auc": auc,
            "delta_macro_f1_vs_stratified_dummy_same_rows": metrics["delta_macro_f1_vs_stratified_dummy"],
            "delta_macro_f1_vs_always_up_dummy_same_rows": metrics["delta_macro_f1_vs_always_up_dummy"],
            "max_ticker_retained_share": float(ticker_counts.max() / retained_n),
            "min_ticker_retained_n": int(ticker_counts.min()),
            "warnings": "|".join(warnings_list),
            "scope": RESULT_SCOPE,
            **{k: metrics[k] for k in ("macro_f1", "balanced_accuracy", "accuracy")},
        })
    return rows


def run_selective_coverage_04c():
    if not OUTPUT_FILES["prediction_manifest"].exists():
        raise FileNotFoundError(f"Prediction manifest missing: {OUTPUT_FILES['prediction_manifest']}")
    rows = []
    for model_name in REAL_MODELS:
        for seed in FRESH_SEEDS:
            model_payload = load_prediction_artifact(model_name, seed)
            stratified_payload = load_prediction_artifact("stratified_dummy", seed)
            always_up_payload = load_prediction_artifact("always_up_dummy", seed)
            rows.extend(selective_rows_for_artifact(model_name, seed, model_payload, stratified_payload, always_up_payload))
    selective = pd.DataFrame(rows)
    selective.to_csv(OUTPUT_FILES["selective"], index=False)
    return selective


def run_gate_decision_04d():
    for name in ("context", "summary", "selective"):
        if not OUTPUT_FILES[name].exists():
            raise FileNotFoundError(f"04D requires {name} artifact: {OUTPUT_FILES[name]}")
    summary = pd.read_csv(OUTPUT_FILES["summary"])
    selective = pd.read_csv(OUTPUT_FILES["selective"])
    rows = []
    for exit_name in ("Exit A - Proceed To 05 LightGBM Tuning", "Exit B - Proceed To 05 MS-DLinear+TCN Design Review", "Exit C - Stop Modeling And Write Weak-Signal Result", "Exit D - Inconclusive, Pre-Register One New Diagnostic"):
        rows.append({
            "exit": exit_name,
            "operator_selected": False,
            "manual_operator_decision_required": True,
            "holdout_test_authorized": False,
            "notes": "Read Notebook 04 summary and selective coverage before manually selecting exactly one exit.",
        })
    lightgbm = summary.loc[summary["model"].eq("lightgbm")]
    msd = summary.loc[summary["model"].eq("ms_dlinear_tcn")]
    tcn = summary.loc[summary["model"].eq("standalone_tcn")]
    checks = {
        "lightgbm_basic_gate": bool((not lightgbm.empty) and bool(lightgbm.iloc[0].get("basic_gate_pass", False))),
        "ms_dlinear_tcn_basic_gate": bool((not msd.empty) and bool(msd.iloc[0].get("basic_gate_pass", False))),
        "ms_dlinear_tcn_not_failed_fresh_seed": bool((not msd.empty) and msd.iloc[0].get("fresh_seed_stability_tag") != "failed_fresh_seed_confirmation"),
        "standalone_tcn_reference_available": bool(not tcn.empty),
        "selective_rows_available": bool(not selective.empty),
        "pre04_design_review_source_required_for_exit_b": True,
    }
    decision = pd.DataFrame(rows)
    for key, value in checks.items():
        decision[key] = value
    decision.to_csv(OUTPUT_FILES["decision"], index=False)
    return decision


def bootstrap_macro_f1(y_true, y_pred, seed):
    rng = np.random.default_rng(seed)
    values = []
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    for _ in range(BOOTSTRAP_RESAMPLES):
        idx = rng.integers(0, len(y_true), size=len(y_true))
        values.append(f1_score(y_true[idx], y_pred[idx], labels=[0, 1], average="macro", zero_division=0))
    return float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))


def run_bootstrap_ci_04e():
    rows = []
    for model_name in REAL_MODELS:
        for seed in FRESH_SEEDS:
            payload = load_prediction_artifact(model_name, seed)
            lower, upper = bootstrap_macro_f1(payload["y_true"], payload["y_pred"], BOOTSTRAP_SEED + int(seed))
            rows.append({
                "model": model_name,
                "seed": int(seed),
                "macro_f1_bootstrap_ci_lower": lower,
                "macro_f1_bootstrap_ci_upper": upper,
                "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
                "bootstrap_seed": BOOTSTRAP_SEED + int(seed),
                "scope": RESULT_SCOPE,
                "interpretation": "diagnostic_only_row_level_bootstrap",
            })
    bootstrap = pd.DataFrame(rows)
    bootstrap.to_csv(OUTPUT_FILES["bootstrap"], index=False)
    return bootstrap


## 04S - Schema Smoke

04S uses tiny synthetic arrays to check the prediction artifact schema,
selective-coverage helper shape, and torch model forward shapes. It does not
load real validation results, fit real data, or write selection decisions.


In [ ]:
\
if RUN_04S_SCHEMA_SMOKE:
    run_notebook04_shape_smoke()
else:
    print("RUN_04S_SCHEMA_SMOKE is False; schema smoke not run.")


## 04A - Read-Only Context Check

04A reads Notebook 03 selection context and optional diagnostic H0 outputs. It
does not fit models.


In [ ]:
\
if RUN_04A_READ_CONTEXT:
    context = run_context_check_04a()
    backup_notebook04_outputs("completed_04A_context_check")
    display(pd.DataFrame([context]))
else:
    print("RUN_04A_READ_CONTEXT is False; context check not run.")


## 04B - Fresh-Seed Confirmation Panel

04B fits the fixed model panel on the fixed official candidate using fresh
seeds. It writes pooled, per-ticker, summary, run-manifest, prediction-manifest,
and per-sample prediction artifacts.


In [ ]:
\
if RUN_04B_FRESH_SEED_PANEL:
    pooled, per_ticker, summary, prediction_manifest = run_fresh_seed_panel_04b()
    display(summary.sort_values(["basic_gate_pass", "macro_f1_mean"], ascending=[False, False]))
    display(prediction_manifest)
else:
    print("RUN_04B_FRESH_SEED_PANEL is False; fresh-seed panel not run.")


## 04C - Within-Model Selective Coverage Diagnostic

04C reads only 04B prediction artifacts. Selective coverage is within-model
only, and retained-subset dummy deltas use the exact retained sample ids.


In [ ]:
\
if RUN_04C_SELECTIVE_COVERAGE:
    selective = run_selective_coverage_04c()
    backup_notebook04_outputs("completed_04C_selective_coverage")
    display(selective)
else:
    print("RUN_04C_SELECTIVE_COVERAGE is False; selective coverage not run.")


## 04D - Manual Gate Decision

04D creates a decision matrix. It does not auto-authorize Notebook 05 or
holdout/test evaluation. The operator must read the matrix and manually select
one exit outside this cell.


In [ ]:
\
if RUN_04D_GATE_DECISION:
    decision = run_gate_decision_04d()
    backup_notebook04_outputs("completed_04D_gate_decision")
    display(decision)
else:
    print("RUN_04D_GATE_DECISION is False; manual gate decision not run.")


## Optional 04E - Bootstrap CI Appendix

04E is off by default. If enabled, it reads only 04B prediction artifacts and
computes diagnostic row-level bootstrap confidence intervals for full-coverage
macro F1.

Important caveat: adjacent sliding-window samples in this time-series setup are
autocorrelated. Row-level bootstrap intervals are therefore diagnostic variance
warnings only, not formal independent-sample inference.


In [ ]:
\
if RUN_04E_BOOTSTRAP_CI:
    bootstrap = run_bootstrap_ci_04e()
    backup_notebook04_outputs("completed_04E_bootstrap_ci")
    display(bootstrap)
else:
    print("RUN_04E_BOOTSTRAP_CI is False; bootstrap CI appendix not run.")


## Interpretation Boundary

Notebook 04 is `validation_only`.

Required interpretation points after a run:

- whether the fixed official Stage 0 candidate passed fresh-seed gates for any
  real model;
- whether each real model was confirmed, marginal, failed fresh-seed
  confirmation, or missing Notebook 03 reference context;
- pooled and per-ticker results;
- dummy baseline deltas on the same target rows;
- selective coverage profiles and ticker concentration warnings;
- sample counts and missing or failed artifact rows;
- explicit statement that holdout/test remains closed.

Allowed wording:

```text
Notebook 04 provides validation-only fresh-seed confirmation and within-model
selective-coverage diagnostics for the fixed official Stage 0 candidate. The
result does not authorize holdout/test evaluation.
```

Forbidden wording:

```text
The best model is ready for holdout.
The model is profitable.
The high-confidence subset is tradable.
Window 32 replaces the official Stage 0 candidate.
Notebook 04 tuned the final model.
The selective curve proves one model is globally better than another.
```
